# 🐭 Remy.IA · Versão final

**Redes Neurais Artificiais e Machine Learning** · Uniso · 6º semestre · Prof. Jaime Ranulfo Leite Filho
**Integrantes:** Beatriz, Davi, Kayke e Guilherme

O usuário diz o que tem na geladeira e o Remy sugere **receitas brasileiras reais**, mostra o que falta comprar, a receita completa e monta um cardápio do dia. Tudo em português.

| Etapa | O que faz | Ferramentas |
|---|---|---|
| 1. Dados | Receitas e **notas reais** do TudoGostoso (coleta autorizada) | pandas |
| 2. Ingredientes | Limpa "2 xícaras (chá) de farinha" → *farinha de trigo*, singular, sinônimos | regras + regex |
| 3. Calorias | Estima kcal por porção somando os ingredientes na **Tabela TACO** (Unicamp) | TACO |
| 4. Embeddings | Acha ingredientes parecidos e corrige o que o usuário digita | Sentence Transformers + FAISS |
| 5. Busca (RAG) | Ranqueia as receitas pelo que você tem, pela nota e pela rapidez | numpy / scipy |
| 6. Cardápio | Café, prato principal, acompanhamento e sobremesa dentro da meta de kcal | regras |
| 7. LLM | Escreve a conversa só com as receitas encontradas | **Qwen3.5-2B** |
| 8. Site | Interface do Davi, ligada a este notebook pelo Gradio | HTML/CSS/JS + Gradio |

## Antes de rodar
1. **Ambiente de execução › Alterar o tipo › GPU T4.**
2. Deixe a base `receitas_tudogostoso.parquet` no seu Drive, na pasta `RemyIA_dados` (o notebook de coleta já salva lá). Se não estiver, o notebook pede para você enviar o arquivo.
3. **Ambiente de execução › Executar tudo.** Leva uns 5 minutos para os dados e uns 3 para o Qwen.
4. No final aparece um link `https://....gradio.live`: cole no site em **Conectar modelo**.

Se aparecer erro de import logo no começo (ex.: `PIL`), use **Ambiente de execução › Reiniciar sessão** e rode tudo de novo.

**Fonte dos dados:** receitas e avaliações do [TudoGostoso](https://www.tudogostoso.com.br) (Webedia), coletadas com autorização para uso acadêmico. Não publique a base. Calorias: Tabela Brasileira de Composição de Alimentos (TACO, NEPA/Unicamp), via [taco-api](https://github.com/raulfdm/taco-api).

In [ ]:
# Instala as bibliotecas · ~1 min
# Não atualizamos Pillow/torchvision: o Colab já traz, e atualizar com a sessão aberta causa erro de import.
!pip install -q -U transformers accelerate
!pip install -q sentence-transformers faiss-cpu gradio pyarrow

In [ ]:
# ============ CONFIGURAÇÃO ============
import os, re, json, math, random, time, unicodedata, warnings
from collections import Counter
from functools import lru_cache
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ARQUIVO_BASE = "/content/drive/MyDrive/RemyIA_dados/receitas_tudogostoso.parquet"
MODELO_LLM = "Qwen/Qwen3.5-2B"                                             # LLM aberto, 2 bi de parâmetros
MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # entende português
USAR_LLM = True          # False = só busca, sem baixar o Qwen (bem mais rápido)

N_RECEITAS = 4           # cards por resposta
MAX_VARIANTES = 3        # variantes do mesmo prato dentro de um card
MAX_FALTANTES = 2        # máximo de ingredientes faltando
TOKENS_LLM = 350         # tamanho máximo do texto do Remy (menor = mais rápido)

GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "NÃO (ative a T4 para o LLM ficar rápido)")

def sem_acento(t):
    return "".join(c for c in unicodedata.normalize("NFD", str(t)) if unicodedata.category(c) != "Mn")

## 1. Dados: receitas do TudoGostoso

A base veio do notebook de coleta (`RemyIA_Coleta_TudoGostoso.ipynb`). Cada receita tem nome, categoria, tempo, porções, **nota e número de avaliações reais**, ingredientes (separados em partes como *Massa* e *Cobertura*) e o modo de preparo.

Por causa da **LGPD**, tiramos a coluna com o nome de quem publicou a receita: o Remy não precisa dela.

A nota usada no ranking é a **nota bayesiana**: uma receita com uma única nota 5 não passa na frente de uma com milhares de notas 4,7.

In [ ]:
# ============ 1. CARREGA A BASE ============
if not os.path.exists(ARQUIVO_BASE):
    try:
        from google.colab import drive, files
        drive.mount("/content/drive")
        if not os.path.exists(ARQUIVO_BASE):
            print("Não achei a base no Drive. Envie o arquivo receitas_tudogostoso.parquet:")
            enviado = files.upload()
            ARQUIVO_BASE = list(enviado)[0]
    except ImportError:
        pass

df = pd.read_parquet(ARQUIVO_BASE)
df = df.drop(columns=[c for c in ("autor", "imagem") if c in df.columns])     # LGPD: sem nomes de pessoas
df = df.drop_duplicates("id").dropna(subset=["nome"]).reset_index(drop=True)
df = df[df["ingredientes"].apply(len) > 0].reset_index(drop=True)

# O site manda alguns textos com códigos HTML ("ficar&aacute;" = "ficará"): voltamos para o texto normal
import html
_limpa = lambda t: re.sub(r"\s+", " ", html.unescape(html.unescape(str(t)))).strip()
df["nome"] = df["nome"].apply(_limpa)
df["ingredientes"] = df["ingredientes"].apply(lambda l: [dict(i, texto=_limpa(i["texto"])) for i in l])
df["passos"] = df["passos"].apply(lambda l: [dict(p, texto=_limpa(p["texto"])) for p in l])
df["categoria"] = df["categoria"].fillna("Outros").str.strip()
df["minutos"] = df["tempo_min"]
df["porcoes"] = df["porcoes"].fillna(df.groupby("categoria")["porcoes"].transform("median")).fillna(4).clip(1, 50)
df["n_avaliacoes"] = df["n_avaliacoes"].fillna(0)

# Nota bayesiana: puxa para a média geral quem tem poucas avaliações (m = 10 avaliações "de mentirinha")
M_BAYES = 10
media_geral = (df["nota"] * df["n_avaliacoes"]).sum() / max(df["n_avaliacoes"].sum(), 1)
df["nota_bayes"] = ((df["nota"].fillna(media_geral) * df["n_avaliacoes"] + media_geral * M_BAYES)
                    / (df["n_avaliacoes"] + M_BAYES))

# Algumas "categorias" com 1 receita são nomes de pratos que o site pôs na trilha: deduzimos pelo nome
POR_NOME = [(r"^sopa|caldo", "Sopas"), (r"^salada|vinagrete", "Saladas, molhos e acompanhamentos"),
            (r"pudim|mousse|charlote|baba de moca|doce|brigadeiro|sorvete", "Doces e sobremesas"),
            (r"^bolo|torta doce", "Bolos e tortas doces"), (r"^pao|lanche|sanduiche", "Lanches"),
            (r"macarrao|lasanha|espaguete|nhoque", "Massas"), (r"camarao|peixe|bacalhau|mexilh|pescador", "Peixes e frutos do mar"),
            (r"frango|galinha", "Aves"), (r".", "Prato Único")]
contagem = df["categoria"].value_counts()
rara = df["categoria"].map(contagem) < 5
df.loc[rara, "categoria"] = df.loc[rara, "nome"].apply(
    lambda n: next(c for p, c in POR_NOME if re.search(p, sem_acento(n.lower()))))

print(f"{len(df):,} receitas · {df['n_avaliacoes'].sum():,.0f} avaliações reais · nota média {media_geral:.2f}")
df[["nome", "categoria", "minutos", "porcoes", "nota", "n_avaliacoes", "nota_bayes"]].head()

## 2. Análise exploratória
Como a base se distribui por categoria, tempo de preparo e nota.

In [ ]:
# ============ 2. ANÁLISE EXPLORATÓRIA ============
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
df["categoria"].value_counts().head(12).sort_values().plot.barh(ax=ax[0], color="#c8794a")
ax[0].set_title("Receitas por categoria"); ax[0].set_xlabel("receitas")
df["minutos"].clip(upper=180).plot.hist(bins=36, ax=ax[1], color="#d9b26a")
ax[1].set_title("Tempo de preparo (até 180 min)"); ax[1].set_xlabel("minutos")
df.loc[df["n_avaliacoes"] > 0, "nota"].plot.hist(bins=20, ax=ax[2], color="#7a9a6a")
ax[2].set_title("Nota dos usuários"); ax[2].set_xlabel("nota (1 a 5)")
plt.tight_layout(); plt.show()

print(f"Tempo mediano: {df['minutos'].median():.0f} min · porções medianas: {df['porcoes'].median():.0f}")
print(f"Receitas com nota: {(df['n_avaliacoes'] > 0).mean():.0%} · ingredientes por receita: "
      f"{df['ingredientes'].apply(len).median():.0f} (mediana)")

## 3. Ingredientes: do texto da receita para o nome do ingrediente

Na receita está escrito *"2 xícaras (chá) de farinha de trigo peneirada"*. Para a busca, interessa **farinha de trigo**. Para as calorias, interessa **2 xícaras ≈ 240 g**. Esta seção separa as duas coisas:

1. **Quantidade e unidade:** números, frações (½, 1/2, "meia"), xícara, colher (sopa/chá/café), lata, caixinha, g, kg, ml...
2. **Nome limpo:** sem acento, sem "picada", "a gosto", "para untar", no singular (*ovos → ovo*) e com sinônimos juntados (*maisena → amido de milho*, *leite moça → leite condensado*).
3. **Alternativas:** *"manteiga ou margarina"* vale com qualquer uma das duas; *"sal e pimenta"* vira dois ingredientes.
4. **Famílias:** quem tem **muçarela** tem **queijo**; quem tem **frango** tem **peito de frango**. Isso evita dizer que falta algo que a pessoa tem.

In [ ]:
# ============ 3. INGREDIENTES ============
# ---------- quantidade e unidade ----------
NUMEROS = {"meia": .5, "meio": .5, "um": 1, "uma": 1, "dois": 2, "duas": 2, "tres": 3, "quatro": 4, "cinco": 5,
           "seis": 6, "sete": 7, "oito": 8, "dez": 10, "doze": 12, "duzia": 12}
FRACOES = {"½": .5, "¼": .25, "¾": .75, "⅓": 1 / 3, "⅔": 2 / 3}
UNIDADES = {   # unidade -> chave usada no cálculo de gramas
    r"x[ií]caras?(?:\s*\(?\s*(?:de\s+)?ch[aá]\s*\)?)?": "xicara", r"copos?\s*(?:\(?\s*(?:tipo\s+)?americano\s*\)?)": "copo americano",
    r"copos?\s*\(\s*(?:de\s+|tipo\s+)?requeij[aã]o\s*\)": "copo requeijao", r"copos?": "copo",
    r"colher(?:es)?\s*\(?\s*(?:de\s+)?sopa\s*\)?": "colher sopa", r"colher(?:es)?\s*\(?\s*(?:de\s+)?ch[aá]\s*\)?": "colher cha",
    r"colher(?:es)?\s*\(?\s*(?:de\s+)?caf[eé]\s*\)?": "colher cafe", r"colher(?:es)?\s*\(?\s*(?:de\s+)?sobremesa\s*\)?": "colher sobremesa",
    r"colherinhas?(?:\s*\(?\s*(?:de\s+)?caf[eé]\s*\)?)?": "colher cha", r"colher(?:es)?": "colher sopa",
    r"latas?": "lata", r"caixas?|caixinhas?": "caixa", r"pacotes?|pct": "pacote", r"vidros?": "vidro", r"potes?": "pote",
    r"envelopes?|sach[eê]s?|saquinhos?": "envelope", r"tabletes?": "tablete", r"barras?": "barra", r"fatias?": "fatia",
    r"dentes?": "dente", r"ma[cç]os?": "maco", r"ramos?|raminhos?": "ramo", r"folhas?": "folha", r"pitadas?": "pitada",
    r"punhados?": "punhado", r"peda[cç]os?": "pedaco", r"conchas?": "concha", r"bandejas?": "bandeja", r"cubos?": "cubo",
    r"quilos?|kg": "kg", r"gramas?|gr?": "g", r"mililitros?|ml": "ml", r"litros?|l": "l", r"unidades?|un": "unidade",
    r"doses?": "dose", r"fios?": "fio", r"gotas?": "gota", r"talos?": "talo", r"cabe[cç]as?": "cabeca", r"postas?": "posta",
    r"fil[eé]s?": "file", r"bifes?": "bife", r"x[ií]caras?\s*\(?\s*(?:de\s+)?caf[eé]\s*\)?": "xicara cafe",
}
_RE_UNIDADE = re.compile(r"^\s*(" + "|".join(f"(?:{k})" for k in UNIDADES) + r")(?![a-zà-ú])\.?\s*(?:de\s+|do\s+|da\s+)?", re.I)
_UNID_ORDEM = list(UNIDADES.items())

def _numero_inicial(t):
    """'1 e 1/2 xícara...' -> (1.5, 'xícara...')"""
    t = t.strip()
    total, achou = 0.0, False
    while True:
        m = re.match(r"^(\d+[.,]\d+|\d+/\d+|\d+\s*[½¼¾⅓⅔]|\d+|[½¼¾⅓⅔])\s*", t)
        if m:
            s = m.group(1).replace(",", ".")
            if "/" in s: a, b = s.split("/"); v = float(a) / float(b) if float(b) else 0
            elif s[-1] in FRACOES: v = float(s[:-1].strip() or 0) + FRACOES[s[-1]]
            else: v = float(s)
            total += v; achou = True; t = t[m.end():]
        else:
            m = re.match(r"^(" + "|".join(NUMEROS) + r")\b\s*", sem_acento(t.lower()))
            if not m: break
            total += NUMEROS[m.group(1)]; achou = True; t = t[m.end():]
        m = re.match(r"^(\d+/\d+)\s*", t)                                   # "2 1/2"
        if m and achou:
            a, b = m.group(1).split("/"); total += float(a) / float(b) if float(b) else 0; t = t[m.end():]
        m = re.match(r"^(e|a|\-|ou)\s+(?=\d|[½¼¾]|meia|meio)", t)          # "1 e 1/2", "2 a 3"
        if not m: break
        if m.group(1) in ("a", "-", "ou"):
            t = re.sub(r"^(a|\-|ou)\s+(\d+[.,]?\d*|[½¼¾])\s*", "", t); break
        t = t[m.end():]
    return (total if achou else None), t

def ler_quantidade(texto):
    """'2 xícaras (chá) de farinha de trigo' -> (2.0, 'xicara', 'farinha de trigo')"""
    t = re.sub(r"\s+", " ", str(texto)).strip()
    t = re.sub(r"^([\d/½¼¾.,\s]+)\((?:uma?|duas?|dois|tres|três|quatro|cinco|seis|meia|meio)\)\s*", r"\1 ", t, flags=re.I)
    qtd, resto = _numero_inicial(t)
    resto = re.sub(r"^de\s+(?=x[ií]cara|colher|copo|lata|caixa|pacote)", "", resto, flags=re.I)   # "2 e 3/4 de xícara"
    unidade = None
    m = _RE_UNIDADE.match(resto)
    if m:
        bruto = m.group(1)
        for padrao, chave in _UNID_ORDEM:
            if re.fullmatch(padrao, bruto, re.I): unidade = chave; break
        resto = resto[m.end():]
        if qtd is None: qtd = 1.0
        m2 = re.match(r"^e\s+(1/2|meia|meio|½)\s+(?:de\s+)?", resto, re.I)          # "1 copo e 1/2 de leite"
        if m2: qtd += .5; resto = resto[m2.end():]
    resto = re.sub(r"^(de|do|da|dos|das)\s+", "", resto.strip(), flags=re.I)
    m = re.match(r"^(?:suco|raspas?)\s+de\s+(\d+|meio|meia|um|uma|dois|duas)\s+", sem_acento(resto.lower()))
    if m:                                                   # "suco de 2 limões"
        qtd = qtd or (NUMEROS.get(m.group(1)) or float(m.group(1)) if not m.group(1).isdigit() else float(m.group(1)))
        unidade = unidade or "unidade"
        resto = resto[:resto.lower().find(" de ") + 4] + resto[m.end():]
    return qtd, unidade, resto

# ---------- nome limpo ----------
DESCRITORES = (r"grandes?|medios?|medias?|pequen[oa]s?|picad[oa]s?|picadinh[oa]s?|ralad[oa]s?|cozid[oa]s?|fatiad[oa]s?|"
               r"cortad[oa]s?|amassad[oa]s?|derretid[oa]s?|amolecid[oa]s?|peneirad[oa]s?|bem|madur[oa]s?|fresc[oa]s?|"
               r"gelad[oa]s?|morn[oa]s?|quentes?|frias?|frios?|filtrad[oa]s?|sem casca|com casca|sem pele|com pele|sem osso|"
               r"descascad[oa]s?|lavad[oa]s?|escorrid[oa]s?|batid[oa]s?|inteir[oa]s?|cheias?|cheios?|rasas?|rasos?|"
               r"graud[oa]s?|aproximadamente|cerca de|opcional|a gosto|q\.?b\.?|de boa qualidade|de sua preferencia|"
               r"temperatura ambiente|em temperatura ambiente|light|ligth|diet|zero|tipo|desfiad[oa]s?|"
               r"limp[oa]s?|congelad[oa]s?|descongelad[oa]s?|tostad[oa]s?|torrad[oa]s?|"
               r"refogad[oa]s?|bem passad[oa]s?|caseir[oa]s?|nacional|importad[oa]s?|organic[oa]s?|"
               r"fervendo|fervente|natural|potavel|o suficiente|suficiente|q b|sem sementes?|separad[oa]s?|firmes?|em neve|passad[oa]s? pela peneira|amanhecid[oa]s?|"
               r"em (?:cubos|cubinhos|tiras|tirinhas|rodelas|pedacos|fatias|bifes|lascas|gomos|metades|quartos)|"
               r"\d+\s*(?:g|kg|ml|l|gramas)\b")
_RE_DESCR = re.compile(rf"\b(?:{DESCRITORES})\b")
CORTE = re.compile(r"\s+(?:para|p/|pra)\s+(?:untar|polvilhar|decorar|fritar|enfeitar|pincelar|servir|acompanhar|o|a)\b.*|"
                   r"\s+se (?:quiser|desejar|preferir|gostar)\b.*|\s+(?:ate|q\.?b)\b.*|\s*[;:].*")

SING_EXCECOES = {"ananas", "anis", "gras", "pires", "lapis", "gas", "mais", "tres", "seis", "brocolis", "pois",
                 "molhos", "nozes", "cuscuz", "arroz", "xadrez", "atras", "deus", "ervas", "lentilhas", "ingles",
                 "frances", "chantilly", "parmesao", "pequis", "kiwis", "tremoco", "chips", "cookies", "nuggets"}
def _singular(w):
    if w in ("chuchus", "abacaxis", "kiwis", "caquis", "pequis"): return w[:-1]
    if len(w) <= 3 or w in SING_EXCECOES: return {"ervas": "erva", "lentilhas": "lentilha", "nozes": "noz", "molhos": "molho"}.get(w, w)
    for a, b in (("eses", "es"), ("oes", "ao"), ("aes", "ao"), ("ais", "al"), ("eis", "el"), ("ns", "m"), ("zes", "z"),
                 ("res", "r")):
        if w.endswith(a): return w[: -len(a)] + b
    return w[:-1] if w.endswith("s") and not w.endswith(("ss", "us", "is")) else w

SINONIMOS = {
    "maisena": "amido de milho", "maizena": "amido de milho", "amido": "amido de milho",
    "leite moca": "leite condensado", "moca": "leite condensado", "trigo": "farinha de trigo",
    "farinha de trigo com fermento": "farinha de trigo", "farinha de trigo especial": "farinha de trigo",
    "leite de vaca": "leite", "leite integral": "leite", "leite desnatado": "leite", "leite semidesnatado": "leite",
    "leite liquido": "leite", "leite fluido": "leite", "creme de leite sem soro": "creme de leite",
    "creme de leite com soro": "creme de leite", "creme de leite fresco": "creme de leite",
    "nata": "creme de leite", "achocolatado em po": "achocolatado", "nescau": "achocolatado", "toddy": "achocolatado",
    "essencia de baunilha": "baunilha", "extrato de baunilha": "baunilha", "essencia baunilha": "baunilha",
    "azeite de oliva": "azeite", "azeite extra virgem": "azeite", "azeite de oliva extra virgem": "azeite",
    "queijo mussarela": "mussarela", "mucarela": "mussarela", "queijo mucarela": "mussarela", "mozarela": "mussarela",
    "mozzarella": "mussarela", "muzzarela": "mussarela", "queijo mozarela": "mussarela", "queijo mozzarella": "mussarela",
    "parmesao": "queijo parmesao", "queijo ralado": "queijo parmesao", "parmesao ralado": "queijo parmesao",
    "requeijao cremoso": "requeijao", "catupiry": "requeijao", "acucar refinado": "acucar", "acucar cristal": "acucar",
    "acucar branco": "acucar", "fermento em po quimico": "fermento em po", "fermento quimico": "fermento em po",
    "fermento quimico em po": "fermento em po", "fermento para bolo": "fermento em po", "fermento royal": "fermento em po",
    "fermento biologico seco": "fermento biologico", "fermento biologico fresco": "fermento biologico",
    "fermento para pao": "fermento biologico", "fermento biologico seco instantaneo": "fermento biologico",
    "oleo de soja": "oleo", "oleo vegetal": "oleo", "oleo de girassol": "oleo", "oleo de canola": "oleo",
    "catchup": "ketchup", "catichup": "ketchup", "clara em neve": "clara", "clara de ovo": "clara", "gema de ovo": "gema",
    "ovo de galinha": "ovo", "banana nanica": "banana", "banana prata": "banana", "banana caturra": "banana",
    "pimenta do reino moida": "pimenta do reino", "pimenta do reino preta": "pimenta do reino", "pimenta preta": "pimenta do reino",
    "salsa": "salsinha", "salsa picada": "salsinha", "cheiro verde picado": "cheiro verde",
    "champion": "champignon", "cogumelo champignon": "champignon", "cogumelo paris": "champignon",
    "milho em conserva": "milho verde", "lata de milho": "milho verde", "milho verde em conserva": "milho verde",
    "margarina sem sal": "margarina", "margarina com sal": "margarina", "manteiga sem sal": "manteiga",
    "manteiga com sal": "manteiga", "caldo de frango": "caldo de galinha", "massa de tomate": "extrato de tomate",
    "polpa de tomate": "molho de tomate", "tomate pelado": "molho de tomate", "passata de tomate": "molho de tomate",
    "agua filtrada": "agua", "agua fervente": "agua", "coco ralado seco": "coco ralado", "coco seco ralado": "coco ralado",
    "cebola roxa": "cebola", "cebola branca": "cebola", "batata inglesa": "batata", "alho picado": "alho",
    "dente de alho": "alho", "farinha de mandioca torrada": "farinha de mandioca", "farinha de mandioca crua": "farinha de mandioca",
    "polvilho doce": "polvilho", "polvilho azedo": "polvilho", "goma de tapioca": "polvilho", "fecula de mandioca": "polvilho",
    "macaxeira": "mandioca", "aipim": "mandioca", "fuba mimoso": "fuba", "farinha de milho flocada": "farinha de milho",
    "flocao": "farinha de milho", "flocos de milho": "farinha de milho", "gelatina incolor": "gelatina sem sabor",
    "gelatina em po sem sabor": "gelatina sem sabor", "arroz agulhinha": "arroz", "arroz branco": "arroz", "arroz tipo 1": "arroz",
    "arroz cru": "arroz", "carne bovina": "carne", "carne de vaca": "carne", "carne de boi": "carne",
    "file de peito de frango": "peito de frango", "peito de frango sem osso": "peito de frango", "biscoito maisena": "biscoito maisena",
    "bolacha maisena": "biscoito maisena", "bolacha maria": "biscoito maisena", "leite em po integral": "leite em po",
    "sal grosso": "sal grosso", "sal fino": "sal", "sal refinado": "sal", "suco de limao": "limao", "suco de 1 limao": "limao",
    "raspa de limao": "limao", "suco de laranja natural": "suco de laranja", "linguica calabresa": "calabresa",
    "linguica tipo calabresa": "calabresa", "pe de alface": "alface", "espiga de milho": "milho verde",
    "sobre coxa": "sobrecoxa de frango", "sobrecoxa": "sobrecoxa de frango", "coxa": "coxa de frango",
    "cebola de cabeca": "cebola", "cubinho de caldo de carne": "caldo de carne", "cubinho de caldo de galinha": "caldo de galinha",
    "requeijao cremoso": "requeijao", "cogumelo": "champignon", "beringela": "berinjela", "milharina": "farinha de milho",
    "coloral": "colorau", "pomarola": "molho de tomate", "pedra de gelo": "gelo", "pau de canela": "canela em pau",
    "caldo de limao": "limao", "polpa de maracuja": "suco de maracuja", "mandioquinha": "batata baroa",
    "queijo provolone": "provolone", "coca cola": "refrigerante", "mesma medida de leite": "leite", "mesma medida de leite de vaca": "leite", "vez a mesma medida de leite": "leite", "salsao": "aipo", "pure de tomate": "molho de tomate", "uva passa": "uva passa", "passa": "uva passa",
    "shoyo": "shoyu", "molho shoyo": "shoyu", "molho shoyu": "shoyu", "molho de soja": "shoyu", "chantily": "chantilly",
    "caldo knnor": "caldo de legume", "caldo knorr": "caldo de legume",
}

# Medidas e recipientes que sobram no começo do nome: "prato fundo de queijo", "sobra de frango", "chá de feijão"
MEDIDA_INICIO = re.compile(r"^\s*(?:(?:um|uma|meio|meia|o|a|os|as|de|do|da|\d+)\s+)*(?:prato (?:fundo|raso)|pratos?|x[ií]caras?|xic|"
                           r"copos?|colher(?:es|inhas?)?|latas?|caixas?|caixinhas?|pacotes?|pacotinhos?|fatias?|pedacos?|"
                           r"pitadas?|punhados?|porc(?:ao|oes)|medidas?|sobras?|restos?|tabletes?|vidros?|potes?|potinhos?|"
                           r"bandejas?|tigelas?|conchas?|saquinhos?|sache?s?|envelopes?|unidades?|cha|cafe|sopa|core|cubos?|"
                           r"cubinhos?|ramos?|macos?|tiras?|rodelas?|gomos?|lascas?|bolinhas?|bolas?|garrafas?|garrafinhas?|pires|calices?|"
                           r"americano|espigas?|formas?|kg|gr?|ml)"
                           r"(?:\s+(?:cheia|cheio|rasa|raso|grande|media|medio|pequena|pequeno|bem|fina|fino|finas|finos|grossa|grosso))*\s+(?:de|do|da|dos|das)\s+")

def canon(nome):
    """Nome do ingrediente -> forma canônica para a busca."""
    t = sem_acento(str(nome).lower())
    t = re.sub(r"[\"'“”‘’*]", " ", t).replace("-", " ").replace("/", " ")
    t = re.sub(r"\(.*?\)|\[.*?\]", " ", t)
    t = t.split(",")[0]
    t = CORTE.sub("", t)
    # "sal para temperar o frango" -> "sal"; mas "massa para pastel" continua igual
    t = re.sub(r"(?<!massa)(?<!farinha)(?<!fermento)(?<!mistura)(?<!tempero)(?<!molho)(?<!caldo)(?<!\bpo)\s+(?:para|p/|pra)\s+.*", "", t)
    t = re.sub(r"\s+(?:a gosto|tambem|se necessario|ja temperad[oa]).*", "", t)
    for _ in range(2): t = MEDIDA_INICIO.sub(" ", t)
    t = _RE_DESCR.sub(" ", t)
    t = re.sub(r"^\s*(?:de|do|da|dos|das|e|ou|com|o|a|os|as)\s+", " ", t)
    t = re.sub(r"\s+(?:de|do|da|dos|das|e|ou|com|em|a)\s*$", "", t.strip())
    palavras = [_singular(w) for w in re.sub(r"[^a-z ]", " ", t).split()]
    t = " ".join(palavras).strip()
    t = SINONIMOS.get(t, t)
    return t

def alternativas(resto):
    """'manteiga ou margarina' -> [['manteiga', 'margarina']]; 'sal e pimenta' -> [['sal'], ['pimenta do reino']]"""
    base = sem_acento(str(resto).lower())
    base = CORTE.sub("", re.sub(r"\(.*?\)", " ", base.split(",")[0]))
    vagas = []
    for parte in re.split(r"\s+e\s+(?!meia|meio)", base):
        opcoes = [canon(p) for p in re.split(r"\s+ou\s+", parte)]
        opcoes = [o for o in dict.fromkeys(opcoes) if o and len(o) > 1]
        if opcoes: vagas.append(opcoes)
    if len(vagas) == 2 and vagas[0] == ["sal"] and vagas[1] == ["pimenta"]:
        vagas[1] = ["pimenta do reino"]
    return vagas

TITULOS_SECAO = re.compile(r"^\s*(recheio|massa|cobertura|calda|montagem|decoracao|decoração|creme|farofa|"
                           r"marinada|tempero|acompanhamento)s?\s*(:|a gosto\b.*)?\s*$|:\s*$", re.I)

def ler_ingrediente(texto):
    if TITULOS_SECAO.search(str(texto)):                          # "Recheio:" é título, não ingrediente
        return {"texto": texto, "qtd": None, "unidade": None, "vagas": []}
    qtd, unidade, resto = ler_quantidade(texto)
    return {"texto": texto, "qtd": qtd, "unidade": unidade, "vagas": alternativas(resto)}

# Lê todos os ingredientes da base
df["itens"] = df["ingredientes"].apply(lambda l: [dict(ler_ingrediente(i["texto"]), secao=i.get("secao")) for i in l])
df["vagas"] = df["itens"].apply(lambda l: [v for i in l for v in i["vagas"]])

contagem_ing = Counter(n for vs in df["vagas"] for v in vs for n in v)
print(f"{len(contagem_ing):,} ingredientes diferentes depois da limpeza")
for t in ["2 xícaras (chá) de farinha de trigo peneirada", "1 lata de leite moça", "3 ovos inteiros", "sal e pimenta a gosto",
          "1 e 1/2 colher (sopa) de manteiga ou margarina", "suco de 2 limões", "½ kg de peito de frango em cubos",
          "1 cebola grande picadinha", "200 g de queijo mussarela ralado", "óleo para fritar"]:
    i = ler_ingrediente(t); print(f"  {t:48s} → {i['qtd']} {i['unidade'] or ''} · {i['vagas']}")
print("\nMais comuns:", ", ".join(f"{n} ({c})" for n, c in contagem_ing.most_common(25)))

## 4. Calorias pela Tabela TACO

O TudoGostoso não informa calorias. Estimamos assim:

1. Cada ingrediente vira **gramas**: *2 xícaras de farinha* ≈ 2 × 120 g; *1 lata de leite condensado* = 395 g; *3 ovos* ≈ 150 g.
2. Cada ingrediente é ligado a um alimento da **TACO** (Tabela Brasileira de Composição de Alimentos, NEPA/Unicamp), que dá as **kcal por 100 g**.
3. Somamos tudo e dividimos pelas **porções** da receita.

Ingredientes *"a gosto"*, temperos e água ficam de fora (quase não têm calorias). Se menos de 60% dos ingredientes forem reconhecidos, a receita fica **sem estimativa**, para não mostrar um número ruim. No site aparece como **≈ 350 kcal por porção**: é uma estimativa, não um valor nutricional oficial.

**O que a TACO não tem:** ketchup, bacon, cream cheese, vinho, licor, sorvete, salsicha, champignon, peito de peru, chantilly, uva passa e mais alguns, e o leite vem sem kcal. Esses valores vêm da tabela oficial do **IBGE** (POF 2008-2009, *Tabelas de Composição Nutricional dos Alimentos Consumidos no Brasil*), listados em `dados/taco/complementos.csv` com o código IBGE de cada item. Os poucos casos aproximados estão marcados na coluna `observacao`.

**Margem de erro:** não foi medida. As medidas caseiras (xícara cheia ou rasa) e as porções informadas pelos autores variam, então o número é uma estimativa.

**De onde vem a tabela:** a TACO (4ª edição) está no repositório, em `dados/taco/` (`food.csv` com os 597 alimentos e `nutrients.csv` com os nutrientes). A cópia foi feita do projeto aberto [taco-api](https://github.com/raulfdm/taco-api) (licença MIT), que converteu a tabela oficial para CSV. Se o repositório não estiver acessível, o notebook baixa direto do taco-api.

In [ ]:
# ============ 4. CALORIAS PELA TACO ============
# A TACO fica numa cópia dentro do repositório (pasta dados/taco). Se o repositório não estiver acessível,
# o notebook usa o projeto original taco-api (licença MIT), de onde a cópia foi feita.
TACO_FONTES = [os.environ.get("TACO_DIR", ""), "dados/taco",
               "https://raw.githubusercontent.com/davileal0702/Remy.IA/main/dados/taco",
               "https://raw.githubusercontent.com/raulfdm/taco-api/main/references/csv"]
def _ler(nome):
    for fonte in TACO_FONTES:
        if not fonte or (not fonte.startswith("http") and not os.path.exists(f"{fonte}/{nome}")): continue
        try:
            tabela = pd.read_csv(f"{fonte}/{nome}")
            print(f"  {nome}: {fonte}")
            return tabela
        except Exception:
            continue
    raise FileNotFoundError(f"Não achei {nome} da TACO em nenhuma fonte: {TACO_FONTES}")
taco = _ler("food.csv").merge(_ler("nutrients.csv")[["foodId", "kcal"]], left_on="id", right_on="foodId")
taco = taco.set_index("id")
KCAL100 = taco["kcal"].to_dict()
print(f"TACO: {len(taco)} alimentos")

# ---------- complementos: o que a TACO não tem (ou deixa sem kcal) ----------
# Vêm da tabela oficial do IBGE (POF 2008-2009, "Tabelas de Composição Nutricional dos Alimentos Consumidos no Brasil"),
# com o código de cada item em dados/taco/complementos.csv. Nada aqui foi digitado de memória.
COMPLEMENTO_FONTES = ["dados/taco",
                      "https://raw.githubusercontent.com/davileal0702/Remy.IA/main/dados/taco",
                      "https://raw.githubusercontent.com/davileal0702/Remy.IA/versao_13.11/dados/taco"]
def _ler_complementos():
    for fonte in COMPLEMENTO_FONTES:
        if not fonte.startswith("http") and not os.path.exists(f"{fonte}/complementos.csv"): continue
        try: return pd.read_csv(f"{fonte}/complementos.csv"), fonte
        except Exception: continue
    return None, None
complementos, _fonte_comp = _ler_complementos()
COMPLEMENTO = {}
if complementos is not None:
    COMPLEMENTO = {r.ingrediente: f"ibge:{r.codigo_ibge}" for r in complementos.itertuples()}
    KCAL100.update({f"ibge:{r.codigo_ibge}": float(r.kcal_100g) for r in complementos.itertuples()})
    print(f"Complementos (IBGE): {len(COMPLEMENTO)} ingredientes, de {_fonte_comp}")
else:
    print("⚠️ Não achei dados/taco/complementos.csv: ingredientes fora da TACO ficam sem estimativa.")

# Ingrediente (nome canônico) -> alimento da TACO (id). Os mais comuns da base, ligados à mão.
MAPA_TACO = {
    "acucar": 494, "acucar mascavo": 493, "acucar demerara": 493, "acucar de confeiteiro": 494, "ovo": 489, "gema": 487,
    "clara": 486, "leite": 458, "leite condensado": 453, "creme de leite": 447, "farinha de trigo": 35, "farinha": 35,
    "farinha de rosca": 34, "farinha de milho": 33, "fuba": 43, "amido de milho": 42, "farinha de mandioca": 122,
    "polvilho": 146, "farinha de arroz": 31, "aveia": 7, "farelo de aveia": 7, "manteiga": 261, "margarina": 263,
    "oleo": 272, "azeite": 260, "azeite de dende": 259, "fermento em po": 513, "fermento biologico": 514, "fermento": 513,
    "cebola": 107, "alho": 82, "tomate": 157, "batata": 92, "batata doce": 89, "cenoura": 110, "mandioca": 130,
    "abobrinha": 71, "abobora": 65, "berinjela": 96, "chuchu": 113, "beterraba": 98, "brocolis": 101, "couve flor": 117,
    "couve": 115, "repolho": 149, "alface": 78, "rucula": 152, "espinafre": 119, "pepino": 142, "pimentao": 144,
    "pimentao verde": 144, "pimentao vermelho": 145, "pimentao amarelo": 143, "vagem": 162, "quiabo": 147,
    "salsinha": 153, "cheiro verde": 153, "cebolinha": 108, "manjericao": 133, "palmito": 139, "ervilha": 560,
    "milho verde": 45, "milho": 44, "arroz": 4, "arroz integral": 2, "macarrao": 40, "espaguete": 40, "lasanha": 38,
    "massa de lasanha": 38, "feijao": 562, "feijao carioca": 562, "feijao preto": 568, "feijao branco": 566,
    "lentilha": 578, "grao de bico": 575, "soja": 581, "queijo": 467, "mussarela": 463, "queijo parmesao": 464,
    "queijo prato": 467, "queijo minas": 461, "requeijao": 468, "ricota": 469, "cream cheese": 468, "iogurte natural": 448,
    "iogurte": 448, "leite em po": 459, "leite de coco": 523, "coco ralado": 590, "coco": 590, "agua de coco": 478,
    "chocolate meio amargo": 498, "chocolate ao leite": 495, "chocolate": 498, "chocolate em po": 491, "achocolatado": 491,
    "cacau em po": 491, "chocolate branco": 495, "doce de leite": 501, "goiabada": 198, "mel": 507, "gelatina": 515,
    "gelatina sem sabor": 515, "maionese": 524, "presunto": 439, "apresuntado": 323, "mortadela": 424, "salame": 443,
    "bacon": 444, "toucinho": 444, "calabresa": 421, "linguica": 421, "linguica de frango": 418, "carne": 352,
    "carne moida": 327, "patinho": 376, "alcatra": 369, "acem": 329, "file mignon": 357, "contra file": 345,
    "maminha": 367, "picanha": 380, "costela": 348, "musculo": 372, "lagarto": 364, "cupim": 354, "fraldinha": 362,
    "carne seca": 385, "charque": 339, "frango": 405, "peito de frango": 409, "file de frango": 409,
    "coxa de frango": 399, "sobrecoxa de frango": 414, "asa de frango": 391, "frango desfiado": 408, "lombo": 433,
    "lombo de porco": 433, "pernil": 436, "bisteca": 427, "costelinha de porco": 431, "carne de porco": 433,
    "peixe": 302, "file de peixe": 302, "merluza": 302, "pescada": 307, "tilapia": 302, "salmao": 316, "atum": 277,
    "sardinha": 319, "bacalhau": 279, "camarao": 285, "banana": 179, "maca": 222, "laranja": 214, "suco de laranja": 215,
    "limao": 218, "morango": 239, "abacaxi": 164, "manga": 231, "mamao": 225, "maracuja": 232, "uva": 256, "kiwi": 207,
    "abacate": 163, "pera": 243, "pessego": 245, "pessego em calda": 245, "ameixa": 172, "ameixa seca": 171,
    "figo": 194, "goiaba": 200, "acai": 168, "uva passa": 256, "amendoim": 558, "castanha de caju": 588,
    "castanha do para": 589, "noz": 597, "amendoa": 587, "gergelim": 593, "linhaca": 594, "pao": 53, "pao frances": 53,
    "pao de forma": 52, "biscoito maisena": 8, "biscoito": 8, "bolacha": 8, "biscoito champanhe": 8, "cream cracker": 13,
    "ketchup": 159, "molho de tomate": 159, "extrato de tomate": 158, "shoyu": 518, "caldo de galinha": 325,
    "caldo de carne": 324, "azeitona": 521, "azeitona preta": 520, "azeitona verde": 521, "champignon": 139,
    "cerveja": 474, "cachaca": 472, "vinho": 474, "cafe": 471, "suco de limao": 219, "chantilly": 522, "glicose": 503,
    "groselha": 503, "tofu": 584, "canjica": 19, "ovo de codorna": 485, "conhaque": 472, "pinga": 472, "rum": 472,
    "aguardente": 472, "vodka": 472, "licor": 472, "licor de cacau": 472, "creme de cebola": 28, "sopa de cebola": 28,
    "molho ingles": 518, "suco de maracuja": 234, "maria mole": 504, "bombom": 495, "aipo": 76, "cereja": 245,
    "gordura vegetal": 272, "guarana": 481, "refrigerante": 480, "semente de linhaca": 594, "farelo de trigo": 7,
    "creme de milho": 28, "provolone": 467, "batata baroa": 87, "suco de uva": 258, "suspiro": 494,
    "massa para pastel": 59, "germen de trigo": 7, "trigo para quibe": 35, "caldo de camarao": 325, "tapioca": 146, "paio": 421, "salsicha": 418, "peru": 426, "kani kama": 302, "frango inteiro": 402,
    # ligações novas para alimentos que a TACO tem
    "shoyu": 518, "caldo de legume": 325, "caldo": 325, "caldo de bacon": 325, "caldo de costela": 325, "talharim": 40,
    "batatinha": 92, "chuchu": 113, "abacaxi": 164, "paezinho": 53, "castanha": 588, "coxao mole": 352, "pure de batata": 91,
    "leite de coco": 523, "creme vegetal": 265, "bis": 495, "sonho de valsa": 495, "raspa de chocolate": 495,
    "seleta de legume": 154, "suco concentrado de maracuja": 234, "molho pronto": 159, "banha": 272,
}
ZERO_KCAL = {"sal", "agua", "gelo", "pimenta do reino", "oregano", "louro", "folha de louro", "cominho", "canela",
             "canela em po", "noz moscada", "cravo", "cravo da india", "colorau", "paprica", "pimenta", "alecrim",
             "tomilho", "vinagre", "adocante", "bicarbonato", "bicarbonato de sodio", "corante", "baunilha", "erva doce",
             "sal grosso", "agua quente", "agua fria", "pimenta calabresa", "hortela", "coentro", "salvia", "acafrao",
             "curry", "gengibre", "raspa de limao", "essencia", "tempero", "sazon", "aji no moto", "pimenta dedo de moca",
             "adocante culinario", "casca de limao", "canela em pau", "pimenta vermelha", "erva fina", "tempero verde",
             "alcaparra", "pimenta do reino branca", "fermento", "mostarda", "molho de pimenta", "pimenta biquinho",
             # temperos e aditivos em quantidade pequena (quase não somam calorias)
             "tempero pronto", "tempero completo", "sazon vermelho", "sazon", "ajinomoto", "adocante forno", "adocante em po",
             "emulsificante", "emustab", "raspa de laranja", "casca de laranja", "pimenta de cheiro", "pimenta malagueta",
             "pimenta branca", "paprica picante", "paprica doce", "cravo em po", "manjerona", "vinagre branco", "sal marinho",
             "papel aluminio", "forma de gelo", "obs", "mais"}
DENSIDADE = {"farinha de trigo": .5, "farinha": .5, "farinha de rosca": .5, "farinha de milho": .4, "fuba": .55,
             "amido de milho": .55, "polvilho": .6, "farinha de mandioca": .6, "aveia": .35, "acucar": .75,
             "acucar mascavo": .65, "acucar de confeiteiro": .5, "achocolatado": .4, "chocolate em po": .4,
             "cacau em po": .4, "coco ralado": .35, "queijo parmesao": .4, "mussarela": .45, "queijo": .45,
             "arroz": .8, "feijao": .8, "lentilha": .8, "grao de bico": .8, "manteiga": .95, "margarina": .95,
             "oleo": .9, "azeite": .9, "mel": 1.4, "leite condensado": 1.3, "doce de leite": 1.3, "uva passa": .6,
             "noz": .5, "amendoim": .6, "castanha de caju": .55, "milho verde": .7, "ervilha": .7, "gelatina": .7,
             "leite em po": .5, "biscoito": .4, "biscoito maisena": .4, "chocolate": .7, "goiabada": 1.1}
EMBALAGEM = {"leite condensado": 395, "creme de leite": 200, "milho verde": 170, "ervilha": 170, "atum": 120,
             "sardinha": 125, "extrato de tomate": 140, "molho de tomate": 340, "leite de coco": 200, "palmito": 300,
             "champignon": 100, "azeitona": 100, "gelatina": 25, "gelatina sem sabor": 12, "coco ralado": 100,
             "fermento biologico": 10, "fermento em po": 10, "caldo de galinha": 10, "caldo de carne": 10,
             "chocolate": 170, "chocolate meio amargo": 170, "chocolate ao leite": 170, "margarina": 200,
             "biscoito": 200, "biscoito maisena": 200, "macarrao": 500, "espaguete": 500, "queijo parmesao": 50,
             "leite em po": 400, "requeijao": 200, "cream cheese": 150, "doce de leite": 400, "goiabada": 300,
             "pao de forma": 500, "massa de lasanha": 500, "mussarela": 400, "presunto": 200, "bacon": 250,
             "suco de laranja": 1000, "leite": 1000, "iogurte natural": 170, "iogurte": 170, "pessego em calda": 450}
POR_UNIDADE = {"ovo": 50, "gema": 18, "clara": 32, "cebola": 130, "tomate": 110, "batata": 150, "batata doce": 200,
               "cenoura": 110, "banana": 100, "limao": 40, "laranja": 150, "maca": 150, "pera": 160, "manga": 300,
               "mamao": 500, "abacaxi": 800, "maracuja": 90, "abacate": 300, "pimentao": 150, "abobrinha": 250,
               "berinjela": 300, "chuchu": 300, "pepino": 200, "beterraba": 150, "peito de frango": 300,
               "file de frango": 120, "coxa de frango": 110, "sobrecoxa de frango": 130, "linguica": 100,
               "calabresa": 250, "salsicha": 50, "pao frances": 50, "pao": 50, "pao de forma": 25, "alho": 5,
               "kiwi": 75, "morango": 15, "uva": 5, "azeitona": 4, "biscoito": 8, "biscoito maisena": 6,
               "bolacha": 8, "goiaba": 150, "pessego": 120, "mandioca": 400, "brocolis": 300, "couve flor": 500,
               "alface": 300, "repolho": 1000, "file de peixe": 150, "bife": 120, "coco": 400, "ovo de codorna": 10,
               "frango": 1500, "coxa de frango": 110, "milho verde": 200, "bombom": 20, "paio": 150, "pimentao": 150,
               "cebolinha": 5, "salsinha": 5, "folha de louro": 0}
GRAMAS_UNIDADE = {"g": 1, "kg": 1000, "ml": 1, "l": 1000, "xicara": 240, "xicara cafe": 50, "copo americano": 190,
                  "copo requeijao": 250, "copo": 240, "colher sopa": 15, "colher cha": 5, "colher cafe": 2.5,
                  "colher sobremesa": 10, "fatia": 20, "dente": 5, "maco": 80, "ramo": 3, "folha": 1, "pitada": .5,
                  "punhado": 30, "pedaco": 50, "concha": 100, "bandeja": 400, "cubo": 10, "dose": 50, "fio": 5,
                  "gota": .05, "talo": 30, "cabeca": 40, "posta": 150, "file": 120, "bife": 120}
VOLUME = {"xicara", "xicara cafe", "copo americano", "copo requeijao", "copo", "colher sopa", "colher cha",
          "colher cafe", "colher sobremesa", "ml", "l"}
EMBALAGENS = {"lata", "caixa", "pacote", "vidro", "pote", "envelope", "tablete", "barra"}

@lru_cache(maxsize=None)
def alimento_taco(nome):
    """Nome canônico -> id da TACO. Primeiro o mapa feito à mão; depois, a primeira palavra do nome."""
    if nome in COMPLEMENTO: return COMPLEMENTO[nome]           # tabela do IBGE para o que a TACO não tem
    if nome in MAPA_TACO: return MAPA_TACO[nome]
    palavras = nome.split()
    for n in range(len(palavras) - 1, 0, -1):                    # "farinha de trigo integral" -> "farinha de trigo"
        parte = " ".join(palavras[:n])
        if parte in COMPLEMENTO: return COMPLEMENTO[parte]
        if parte in MAPA_TACO: return MAPA_TACO[parte]
    alvo = palavras[0] if palavras else ""
    candidatos = [i for i, n in taco["name"].items() if sem_acento(n.lower()).split(",")[0].strip() == alvo]
    crus = [i for i in candidatos if re.search(r"\bcru", sem_acento(taco.at[i, "name"].lower()))]
    return (crus or candidatos or [None])[0]

def gramas(item, nome):
    qtd, un = item["qtd"], item["unidade"]
    if qtd is None: return None
    if un is None or un == "unidade":
        for chave in (nome, " ".join(nome.split()[:2]), nome.split()[0]):      # "pimentao verde" -> "pimentao"
            if chave in POR_UNIDADE: return qtd * POR_UNIDADE[chave]
        return None
    if un in EMBALAGENS:
        return qtd * EMBALAGEM.get(nome, 200)
    base = GRAMAS_UNIDADE.get(un)
    if base is None: return None
    return qtd * base * (DENSIDADE.get(nome, .9) if un in VOLUME and un not in ("ml", "l") else 1)

def estimar_kcal(itens, porcoes):
    total, contados, considerados = 0.0, 0, 0
    for item in itens:
        if not item["vagas"]: continue
        nome = item["vagas"][0][0]                                # primeira opção da primeira vaga
        if nome in ZERO_KCAL or item["qtd"] is None and nome not in MAPA_TACO and nome not in COMPLEMENTO: continue
        considerados += 1
        g, alimento = gramas(item, nome), alimento_taco(nome)
        if g is None or alimento is None or pd.isna(KCAL100.get(alimento)): continue
        total += g * KCAL100[alimento] / 100; contados += 1
    cobertura = contados / considerados if considerados else 0
    if cobertura < 0.6 or not porcoes: return None, cobertura
    por_porcao = total / porcoes
    return (round(por_porcao) if 10 <= por_porcao <= 2500 else None), cobertura

estimativas = df.apply(lambda r: estimar_kcal(r["itens"], r["porcoes"]), axis=1)
df["kcal_porcao"] = estimativas.str[0]
df["kcal_cobertura"] = estimativas.str[1]
print(f"Receitas com estimativa de calorias: {df['kcal_porcao'].notna().mean():.0%} "
      f"(mediana {df['kcal_porcao'].median():.0f} kcal por porção)")
df.loc[df["kcal_porcao"].notna(), ["nome", "porcoes", "kcal_porcao", "kcal_cobertura"]].sample(8, random_state=1)

## 5. Vocabulário, embeddings e o que o usuário digita

- **Vocabulário:** todos os ingredientes da base, já limpos.
- **Embeddings** (`paraphrase-multilingual-MiniLM-L12-v2`, uma rede neural que entende português) + **FAISS**: transformam cada ingrediente num vetor. Quando o usuário digita algo que não está no vocabulário, o Remy acha o ingrediente mais parecido.
- **Erros de digitação:** *frnago → frango*, *cbola → cebola* (distância de edição).
- **Famílias:** *frango* também vale para *peito de frango*; *muçarela* também vale para uma receita que pede só *queijo*.
- **Ingredientes ambíguos:** *farinha*, *fermento*, *queijo*, *feijão*, *chocolate*, *milho* e *pimenta* têm tipos que mudam a receita. O Remy usa o tipo mais comum e **pergunta** qual é.
- **Básicos:** sal, água, óleo e pimenta-do-reino não contam como "falta".

In [ ]:
# ============ 5. VOCABULÁRIO, EMBEDDINGS E INTERPRETAÇÃO ============
from sentence_transformers import SentenceTransformer
import faiss

vocab = sorted(contagem_ing)
idx = {n: i for i, n in enumerate(vocab)}
freq = np.array([contagem_ing[n] for n in vocab])

# Nome bonito (com acento) de cada ingrediente, para mostrar no site: a grafia mais usada nas próprias receitas
_grafias = Counter(w for itens in df["itens"] for item in itens
                   for w in re.findall(r"[a-zà-ú]+", str(item["texto"]).lower()))
GRAFIA = {}
for w, n in _grafias.most_common():
    GRAFIA.setdefault(sem_acento(w), w)
EXIBIR = {"mussarela": "muçarela"}
def nome_exibicao(c):
    return EXIBIR.get(c) or " ".join(GRAFIA.get(w, w) for w in c.split())

MIN_RECEITAS_SITE = 5
BASICOS = {"sal", "agua", "gelo", "oleo", "pimenta do reino", "agua quente", "agua fria", "sal grosso"}
DESPENSA = {"acucar", "farinha de trigo", "alho", "cebola", "azeite", "vinagre", "manteiga", "margarina",
            "fermento em po", "oregano", "louro", "folha de louro", "colorau", "caldo de galinha", "caldo de carne", "ovo"}

# Lista para o "Toque para adicionar" do site: ingredientes que aparecem em pelo menos 5 receitas
# (abaixo disso aparecem erros de digitação e frases que não são ingredientes)
RUIDO_SITE = re.compile(r"\b(gosto|receita|mesma|medida|sobra|prato|para|ponto|qualquer|preferencia|tipo|marca|cerca|"
                        r"opcional|enfeitar|decorar|untar|polvilhar|aproximadamente|massa|recheio|cobertura|calda|mais|menos|meno|fogao|forno|"
                        r"panela|forma|assadeira|liquidificador|batedeira|semente|tempero|ingrediente)\b")
INGREDIENTES_SITE = [nome_exibicao(n) for n in sorted(vocab, key=lambda n: -contagem_ing[n])
                     if contagem_ing[n] >= MIN_RECEITAS_SITE and 2 < len(n) <= 30 and len(n.split()) <= 4
                     and n not in BASICOS - {"oleo"} and not RUIDO_SITE.search(n)]
print(f"{len(INGREDIENTES_SITE):,} ingredientes no 'Toque para adicionar' do site, ex.: {INGREDIENTES_SITE[:12]}")

# ---------- famílias ----------
PAIS = {"mussarela": "queijo", "queijo parmesao": "queijo", "queijo prato": "queijo", "queijo minas": "queijo",
        "provolone": "queijo", "cheddar": "queijo", "queijo coalho": "queijo", "patinho": "carne", "alcatra": "carne",
        "acem": "carne", "contra file": "carne", "file mignon": "carne", "maminha": "carne", "musculo": "carne",
        "fraldinha": "carne", "picanha": "carne", "cupim": "carne", "lagarto": "carne", "coxao mole": "carne",
        "coxao duro": "carne", "carne moida": "carne", "espaguete": "macarrao", "penne": "macarrao",
        "talharim": "macarrao", "parafuso": "macarrao", "fusilli": "macarrao", "macarrao parafuso": "macarrao",
        "tilapia": "peixe", "merluza": "peixe", "pescada": "peixe", "salmao": "peixe", "file de peixe": "peixe",
        "calabresa": "linguica", "farinha de trigo": "farinha", "fermento em po": "fermento",
        "fermento biologico": "fermento", "milho verde": "milho"}
DERIVADO = re.compile(r"\b(molho|caldo|extrato|suco|essencia|creme|doce|pure|po|farinha|oleo|polpa|geleia|sopa|calda|"
                      r"xarope|licor|refresco|bala|biscoito|bolacha|pasta|recheio|massa|cobertura|tablete|sorvete|"
                      r"iogurte|leite|condensado|amido|palha|poro|agua|vinagre|tempero|chips|salgadinho|flocos?|gelatina|"
                      r"pudim|mistura|bolo|pao|torta|semente|folha|branco)\b")

def _contem(geral, especifico):
    """'frango' está contido em 'peito de frango' (sem ser derivado, como 'caldo de frango')?"""
    if geral == especifico or not re.search(rf"\b{re.escape(geral)}\b", especifico): return False
    return not DERIVADO.search(especifico.replace(geral, " "))

FILHOS = {n: set() for n in vocab}
for n in vocab:
    for g in set(n.split()) | {" ".join(n.split()[:2])}:
        if g in FILHOS and _contem(g, n): FILHOS[g].add(n)
for filho, pai in PAIS.items():
    if pai in FILHOS and filho in idx: FILHOS[pai].add(filho)

# ---------- ingredientes ambíguos: o que vale para o nome genérico ----------
AMBIGUOS = {   # termo genérico -> (o que o Remy considera, opções para o usuário, o que entra na busca)
    "farinha": ("farinha de trigo", ["farinha de trigo", "farinha de mandioca", "polvilho", "fubá", "farinha de milho",
                "farinha de arroz", "farinha de rosca"], {"farinha", "farinha de trigo"}),
    "fermento": ("fermento químico (em pó)", ["fermento químico", "fermento biológico"], {"fermento", "fermento em po"}),
    "queijo": ("queijos de fatiar e ralar (muçarela, prato, parmesão...)",
               ["muçarela", "queijo prato", "parmesão", "queijo minas", "requeijão", "cream cheese", "ricota"], None),
    "feijao": ("feijão em grão", ["feijão preto", "feijão carioca", "feijão branco", "feijão fradinho"], None),
    "chocolate": ("chocolate em barra (ao leite ou meio amargo)", ["chocolate ao leite", "chocolate meio amargo",
                  "chocolate branco", "chocolate em pó", "gotas de chocolate"], None),
    "milho": ("milho verde", ["milho verde", "fubá", "amido de milho", "pipoca", "canjica"], {"milho", "milho verde"}),
    "pimenta": ("pimenta vermelha", ["pimenta-do-reino", "pimenta dedo-de-moça", "pimenta calabresa", "pimentão"], None),
}
NAO_GENERICO = {"queijo": r"requeijao|cream cheese|ricota|cottage|gorgonzola|catupiry", "chocolate": r"branco",
                "batata": r"doce|palha|baroa", "alho": r"poro", "leite": r"condensado|coco|po|evaporado"}

# ---------- embeddings + FAISS ----------
t0 = time.time()
emb_model = SentenceTransformer(MODELO_EMBEDDINGS)
vetores = emb_model.encode(vocab, batch_size=256, normalize_embeddings=True, show_progress_bar=False).astype("float32")
indice = faiss.IndexFlatIP(vetores.shape[1]); indice.add(vetores)
print(f"Embeddings de {len(vocab):,} ingredientes em {time.time() - t0:.0f}s")

def vizinhos(termo, k=5):
    q = emb_model.encode([termo], normalize_embeddings=True).astype("float32")
    s, i = indice.search(q, k)
    return [(vocab[j], float(v)) for j, v in zip(i[0], s[0])]

# ---------- erros de digitação ----------
def _distancia(a, b):
    d = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        ant, d[0] = d[:], i
        for j in range(1, len(b) + 1):
            d[j] = min(ant[j] + 1, d[j - 1] + 1, ant[j - 1] + (a[i - 1] != b[j - 1]))
            if i > 1 and j > 1 and a[i - 1] == b[j - 2] and a[i - 2] == b[j - 1]:
                d[j] = min(d[j], ant2[j - 2] + 1)
        ant2 = ant
    return d[-1]

@lru_cache(maxsize=4096)
def corrigir(termo):
    """Erro de digitação -> palavra do vocabulário (mais usada em caso de empate)."""
    if len(termo) < 4: return None
    limite = 1 if len(termo) <= 7 else 2
    cands = [(_distancia(termo, v), -contagem_ing[v], v) for v in vocab
             if v[0] == termo[0] and abs(len(v) - len(termo)) <= limite]
    cands = [c for c in cands if c[0] <= limite]
    return min(cands)[2] if cands else None

RARO = 2          # ingrediente que aparece em até 2 receitas pode ser erro de digitação de alguém ("cbola")
def corrigir_raro(termo):
    """Se o termo existe mas é raro e há um vizinho de 1 letra 20x mais usado, fica com o vizinho."""
    if len(termo) < 4 or contagem_ing.get(termo, 0) > RARO: return None
    cands = [(-contagem_ing[v], v) for v in vocab if v != termo and v[0] == termo[0] and abs(len(v) - len(termo)) <= 1
             and contagem_ing[v] >= 20 * max(contagem_ing.get(termo, 1), 1) and _distancia(termo, v) <= 1]
    return min(cands)[1] if cands else None

# ---------- o que o usuário digitou -> ingredientes da base ----------
def separar_itens(texto):
    return [p.strip() for p in re.split(r",|;|\n|\s+e\s+|\+", str(texto)) if p.strip()]

def expandir(c):
    """Nome canônico -> conjunto de ingredientes do vocabulário que ele cobre."""
    if c in AMBIGUOS and AMBIGUOS[c][2] is not None:
        alvos = {n for n in AMBIGUOS[c][2] if n in idx}
    else:
        alvos = ({c} if c in idx else set()) | FILHOS.get(c, set())
        if c in NAO_GENERICO:
            alvos = {n for n in alvos if n == c or not re.search(NAO_GENERICO[c], n)}
    pai = PAIS.get(c) or next((p for p in FILHOS if c in FILHOS[p] and p in PAIS.values()), None)
    if pai and pai in idx: alvos.add(pai)                      # muçarela também vale para "queijo"
    return {idx[n] for n in alvos}

def interpretar(texto):
    """Lista de termos: {'digitado', 'nome', 'ids', 'aviso', 'ambiguo'}"""
    termos = []
    for item in separar_itens(texto):
        c = canon(item)
        termo = {"digitado": item, "nome": item.strip().lower(), "ids": set(), "aviso": None, "ambiguo": None}
        if (c in idx or c in AMBIGUOS or FILHOS.get(c)) and not (cor := corrigir_raro(c)):
            termo["ids"] = expandir(c)
        elif c in idx and cor:
            termo["ids"], termo["nome"] = expandir(cor), cor
            termo["aviso"] = f"✏️ Entendi **{cor}** onde você escreveu *{item}*."
        elif (cor := corrigir(c)):
            termo["ids"], termo["nome"] = expandir(cor), cor
            termo["aviso"] = f"✏️ Entendi **{cor}** onde você escreveu *{item}*."
        else:
            (melhor, sim), *resto = vizinhos(c, 3)
            if sim >= 0.80:
                termo["ids"], termo["nome"] = expandir(melhor), melhor
                termo["aviso"] = f"✏️ Não achei *{item}* nas receitas; usei **{melhor}**, que é parecido."
            elif sim >= 0.60:
                termo["aviso"] = f"🤔 Não achei *{item}*. Você quis dizer **{melhor}**?"
            else:
                termo["aviso"] = f"❓ Não encontrei *{item}* em nenhuma receita."
        chave = canon(termo["nome"])
        if chave in AMBIGUOS:
            termo["ambiguo"] = chave
        termos.append(termo)
    return termos

for t in ["frango", "farinha", "queijo", "mussarela", "frnago", "cbola", "leite moça", "feijão"]:
    r = interpretar(t)[0]
    print(f"{t:12s} → {len(r['ids']):3d} ingredientes, ex.: {[vocab[i] for i in list(r['ids'])[:4]]} {r['aviso'] or ''}")

## 6. Busca (o "R" do RAG)

Cada receita vira uma lista de **vagas** de ingrediente (uma vaga pode ter alternativas, como *manteiga ou margarina*). Montamos uma **matriz esparsa vaga × ingrediente** e, com uma multiplicação, sabemos quais vagas o usuário cobre em todas as receitas de uma vez.

**Pontuação de cada receita:**

`0,45 × cobertura + 0,35 × ingredientes do usuário usados + 0,15 × nota bayesiana / 5 + 0,05 × rapidez − 0,03 × ingredientes faltando`

- **cobertura:** quanto da receita você já tem (os básicos não contam);
- **usados:** quanto da sua lista a receita aproveita;
- **nota bayesiana:** a nota real dos usuários do TudoGostoso, ajustada pelo número de avaliações;
- **rapidez:** receitas mais rápidas ganham um pouco;
- **faltando:** cada ingrediente que falta tira um pouco (até 10), para que, mesmo com *Pode faltar: tanto faz*, as receitas mais fáceis de completar venham primeiro.

**Filtros:** ingredientes faltando (a opção *Pode faltar* do site: nada, até 2, até 5 ou tanto faz), tempo, categoria, vegetariano, vegano, sem glúten, sem lactose, calorias por porção, nota mínima, ingredientes a evitar e "tenho o básico" (açúcar, farinha, alho, cebola, ovo...).

In [ ]:
# ============ 6. BUSCA ============
# ---------- matriz vaga × ingrediente ----------
linhas, colunas, vaga_receita, vaga_item, vaga_basica = [], [], [], [], []
for r, itens in enumerate(df["itens"]):
    for k, item in enumerate(itens):
        for opcoes in item["vagas"]:
            v = len(vaga_receita)
            for n in opcoes:
                linhas.append(v); colunas.append(idx[n])
            vaga_receita.append(r); vaga_item.append(k)
            vaga_basica.append(all(n in BASICOS for n in opcoes))
VAGAS = sp.csr_matrix((np.ones(len(linhas), np.float32), (linhas, colunas)), shape=(len(vaga_receita), len(vocab)))
vaga_receita, vaga_item, vaga_basica = np.array(vaga_receita), np.array(vaga_item), np.array(vaga_basica)
N = len(df)
K_RECEITA = np.bincount(vaga_receita, weights=~vaga_basica, minlength=N)       # vagas que contam (sem os básicos)
print(f"Matriz: {VAGAS.shape[0]:,} vagas × {VAGAS.shape[1]:,} ingredientes · {VAGAS.nnz:,} ligações")

# ---------- restrições de dieta (por ingrediente) ----------
def _mascara(regex, ok=None):
    k = re.compile(ok) if ok else None
    marca = np.array([bool(re.search(regex, k.sub(" ", n) if k else n)) for n in vocab], np.float32)
    return np.bincount(vaga_receita, weights=(VAGAS @ marca) > 0, minlength=N) > 0
CARNES = (r"frango|carne|bacon|presunto|calabresa|linguica|peixe|atum|sardinha|camarao|bacalhau|salmao|peru|lombo|"
          r"costela|pernil|salsicha|salame|mortadela|patinho|alcatra|acem|picanha|maminha|musculo|cupim|fraldinha|"
          r"lagarto|coxao|caldo de galinha|paio|toucinho|kani|charque|figado|moela|lula|polvo|marisco|mexilhao|siri|"
          r"caranguejo|tilapia|merluza|pescada|anchova|file mignon|contra file|hamburguer|almondega|torresmo")
ANIMAL = CARNES + r"|ovo|gema|clara|leite|queijo|manteiga|creme de leite|requeijao|iogurte|\bmel\b|mussarela|parmesao|" \
                  r"cream cheese|ricota|chantilly|nata|gelatina"
GLUTEN = r"farinha de trigo|^farinha$|trigo|\bpao\b|macarrao|espaguete|lasanha|biscoito|bolacha|farinha de rosca|cerveja|" \
         r"aveia|cevada|centeio|massa|penne|talharim|nhoque"
LACTOSE = r"\bleite\b|queijo|manteiga|creme de leite|requeijao|iogurte|mussarela|parmesao|cream cheese|ricota|chantilly|" \
          r"\bnata\b|doce de leite|catupiry|coalhada|kefir|lacteo|\bmoca\b|ninho|cheddar|provolone|gorgonzola|mascarpone|" \
          r"brie|camembert|cottage|polenguinho|petit suisse|sorvete|soro de leite|whey|ganache"
# Versões sem o componente: "leite de coco", "manteiga de amendoim", "leite sem lactose", "queijo vegano"...
SEM_LACTOSE_OK = r"leite (?:de (?:coco|soja|amendoa|amendoas|aveia|arroz|castanha|castanhas|caju|amendoim|gergelim|inhame)|vegetal|" \
                 r"sem lactose|zero lactose|livre de lactose)|(?:sem|zero|livre de) lactose|manteiga (?:de (?:amendoim|cacau|karite)|vegetal|vegana)|" \
                 r"queijo (?:vegano|vegetal|de castanha|de castanhas|de caju|de tofu)|creme de leite (?:de coco|vegetal|sem lactose)|" \
                 r"iogurte (?:de (?:coco|soja)|vegetal|vegano)|requeijao vegano"
SEM_GLUTEN_OK = r"(?:sem|livre de|zero) gluten|farinha de (?:arroz|mandioca|milho|amendoa|amendoas|coco|grao de bico|banana|aveia sem gluten)"

# Para restrições de saúde, o Remy é conservador: olha os ingredientes E também o nome e o modo de preparo
# (tem receita que esconde ingrediente no passo a passo, como "acrescente 15 colheres de leite em pó")
def _texto_receita(i):
    partes = [df.at[i, "nome"]] + [it["texto"] for it in df.at[i, "itens"]] + [p["texto"] for p in df.at[i, "passos"]]
    return sem_acento(" | ".join(map(str, partes)).lower())
TEXTO_RECEITA = [_texto_receita(i) for i in range(N)]
def _mascara_texto(regex, ok=None):
    r, k = re.compile(rf"\b(?:{regex})"), re.compile(ok) if ok else None      # \b: "ovo" não pega "novo"
    return np.array([bool(r.search(k.sub(" ", t) if k else t)) for t in TEXTO_RECEITA])

# No texto livre, "pão de queijo" e "massa" (de qualquer coisa) não indicam glúten
GLUTEN_TEXTO = r"farinha de trigo|trigo|farinha de rosca|macarrao|espaguete|lasanha|talharim|penne|nhoque|biscoito|bolacha|" \
               r"cerveja|cevada|centeio|aveia|pao(?! de queijo)|paes|torrada|massa folhada|massa de pastel|massa pronta|cuscuz marroquino"
TEM_DIETA = {"vegetariano": ~(_mascara(CARNES) | _mascara_texto(CARNES)),
             "vegano": ~(_mascara(ANIMAL, SEM_LACTOSE_OK) | _mascara_texto(ANIMAL, SEM_LACTOSE_OK)),
             "sem glúten": ~(_mascara(GLUTEN, SEM_GLUTEN_OK) | _mascara_texto(GLUTEN_TEXTO, SEM_GLUTEN_OK)),
             "sem lactose": ~(_mascara(LACTOSE, SEM_LACTOSE_OK) | _mascara_texto(LACTOSE, SEM_LACTOSE_OK))}
for d, m in TEM_DIETA.items(): print(f"  {d:12s}: {m.sum():,} receitas")
CATEGORIAS = sorted(df["categoria"].unique())

def _vetor(ids):
    u = np.zeros(len(vocab), np.float32)
    if ids: u[list(ids)] = 1
    return u

def buscar(termos, k=N_RECEITAS, max_faltantes=MAX_FALTANTES, tempo_max=None, categorias=None, dietas=(),
           kcal_max=None, nota_min=0.0, excluir=(), tenho_despensa=False, pular=0, retornar_diag=False):
    """termos: saída de interpretar(). Devolve as melhores receitas com 'tem', 'falta' e a pontuação."""
    validos = [t for t in termos if t["ids"]]
    ids_usuario = set().union(*(t["ids"] for t in validos)) if validos else set()
    hit_usuario = (VAGAS @ _vetor(ids_usuario)) > 0
    hit = hit_usuario | vaga_basica
    if tenho_despensa:
        ids_desp = {i for n in DESPENSA for i in expandir(n)}
        hit = hit | ((VAGAS @ _vetor(ids_desp)) > 0)
    conta = ~vaga_basica
    cobertos = np.bincount(vaga_receita, weights=hit & conta, minlength=N)
    usados = np.bincount(vaga_receita, weights=hit_usuario & conta, minlength=N)
    faltam = K_RECEITA - cobertos

    filtros = {"ingredientes faltando": faltam <= max_faltantes, "usar o que você tem": usados >= 1}
    if tempo_max: filtros["tempo"] = df["minutos"].fillna(9999).to_numpy() <= tempo_max
    if categorias: filtros["categoria"] = df["categoria"].isin(categorias).to_numpy()
    for d in dietas or ():
        if d in TEM_DIETA: filtros[d] = TEM_DIETA[d]
    if kcal_max: filtros["calorias"] = df["kcal_porcao"].fillna(9999).to_numpy() <= kcal_max
    if nota_min: filtros["nota"] = df["nota_bayes"].to_numpy() >= nota_min
    if excluir:
        ids_ex = set().union(*(t["ids"] for t in excluir if t["ids"])) if any(t["ids"] for t in excluir) else set()
        tem_ex = np.zeros(N, bool)
        if ids_ex:
            tem_ex |= np.bincount(vaga_receita, weights=(VAGAS @ _vetor(ids_ex)) > 0, minlength=N) > 0
        # Alergia: também procura a palavra no nome e no modo de preparo ("sirva com amendoim", "Torta de camarão")
        palavras = {sem_acento(str(t.get("digitado") or t["nome"]).lower()).strip() for t in excluir}
        palavras = [w for w in palavras if len(w) >= 3]
        if palavras:
            r_ex = re.compile(r"\b(?:" + "|".join(re.escape(w) + "(?:s|es)?" for w in palavras) + r")\b")
            tem_ex |= np.array([bool(r_ex.search(t)) for t in TEXTO_RECEITA])
        if tem_ex.any(): filtros["ingredientes a evitar"] = ~tem_ex

    ok = np.ones(N, bool)
    for f in filtros.values(): ok &= f
    K = np.maximum(K_RECEITA, 1)
    minutos = df["minutos"].fillna(60).to_numpy()
    score = (0.45 * cobertos / K + 0.35 * usados / max(len(validos), 1)
             + 0.15 * df["nota_bayes"].to_numpy() / 5 + 0.05 * (1 - np.minimum(minutos, 180) / 180)
             - 0.03 * np.minimum(faltam, 10))                   # cada ingrediente faltando pesa contra

    cand = np.where(ok)[0]
    cand = cand[np.argsort(-score[cand])]
    escolhidos, nomes = [], set()
    for i in cand:                                             # tira receitas com o mesmo nome
        chave = sem_acento(df.at[i, "nome"].lower())
        if chave in nomes: continue
        nomes.add(chave); escolhidos.append(i)
        if len(escolhidos) == k + pular: break
    escolhidos = escolhidos[pular:]

    res = df.loc[escolhidos].copy()
    res["score"] = score[escolhidos]; res["cobertura"] = (cobertos / K)[escolhidos]
    tem, falta = [], []
    for i in escolhidos:
        minhas = np.where(vaga_receita == i)[0]
        cobertas = {vaga_item[v] for v in minhas if hit[v]}
        faltando = sorted({vaga_item[v] for v in minhas if not hit[v] and not vaga_basica[v]} - cobertas)
        falta.append([df.at[i, "itens"][j]["texto"] for j in faltando])
        meus = sorted({vaga_item[v] for v in minhas if hit_usuario[v] and not vaga_basica[v]})
        tem.append([df.at[i, "itens"][j]["texto"] for j in meus])        # com quantidade: "3 ovos", "2 copos de leite"
    res["tem"], res["falta"] = tem, falta
    res["rid"] = escolhidos                                  # posição da receita na base (para checar quantidades)
    res = res.reset_index(drop=True)
    if not retornar_diag: return res
    diag = {}
    for nome in filtros:
        sem = np.ones(N, bool)
        for outro, f in filtros.items():
            if outro != nome: sem &= f
        diag[nome] = int(sem.sum())
    return res, diag

teste = buscar(interpretar("frango, arroz, tomate, cebola, alho"), k=5)
teste[["nome", "categoria", "minutos", "nota", "n_avaliacoes", "kcal_porcao", "score", "tem", "falta"]]

## 7. Variantes e cardápio do dia

**Variantes:** o TudoGostoso tem muitas versões do mesmo prato (*Bolo de cenoura*, *Bolo de cenoura fofinho*, *Bolo de cenoura da vovó*...). Elas viram **um card só**, e o usuário escolhe a variante. O nome do prato é o nome da receita sem palavras como *fácil, fofinho, da vovó, de liquidificador* e sem o que vem depois de *com*.

**Cardápio do dia:** café da manhã, almoço, acompanhamento, jantar e sobremesa, usando as categorias do site, dentro da meta de calorias (por porção) e sem repetir o prato.

| Refeição | Categorias |
|---|---|
| Café da manhã | Bolos e tortas doces, Lanches, Bebidas |
| Almoço | Carnes, Aves, Peixes e frutos do mar, Massas, Prato Único |
| Acompanhamento | Saladas, molhos e acompanhamentos, Sopas |
| Jantar | as mesmas do almoço, mais Sopas (sem repetir o prato do almoço) |
| Sobremesa | Doces e sobremesas |

In [ ]:
# ============ 7. VARIANTES E CARDÁPIO ============
ADJETIVOS = set("""facil faceis simples rapido rapida rapidinho rapidinha caseiro caseira perfeito perfeita delicioso
deliciosa gostoso gostosa especial light diet fit fofinho fofinha fofo fofa molhadinho molhadinha super mega top
economico economica saudavel maravilhoso maravilhosa divino divina incrivel tradicional classico classica pratico
pratica sensacional irresistivel cremoso cremosa crocante perfeitinho sequinho sequinha melhor diferente famoso famosa
original verdadeiro verdadeira legitimo legitima basico basica nutritivo nutritiva levinho leve""".split())
EXPRESSOES = re.compile(r"\b(da|do|de) (vovo|vo|mae|mamae|tia|sogra|vizinha|casa|fazenda|roca)\b|\bde liquidificador\b|"
                        r"\bde panela( de pressao)?\b|\bsem forno\b|\b(no|de|em|ao) micro ?ondas\b|\b(ii|iii|iv|2|3)$|\bna airfryer\b|\bsem gluten\b|"
                        r"\bsem lactose\b|\bsem acucar\b|\blow carb\b|\bfit\b|\bzero\b|\bjeito\b.*|\bque\b.*|\bpara\b.*|"
                        r"\bmeu\b|\bminha\b|\bnosso\b|\bnossa\b|\bdo [a-z]+$")

def sem_dono(nome):
    """'Bolo de cenoura da Taty' -> 'Bolo de cenoura' (só quando o resto do nome está em minúsculas)."""
    m = re.search(r"\s+(da|do|de|dos|das)\s+[A-ZÀ-Ú][\wÀ-ú]+(\s+[A-ZÀ-Ú][\wÀ-ú]+)*\s*$", str(nome))
    antes = str(nome)[:m.start()].split()[1:] if m else []
    if m and any(w[:1].islower() and w not in ("de", "do", "da", "com", "e") for w in antes):
        return str(nome)[:m.start()]
    return str(nome)

def nome_base(nome):
    """'Bolo de cenoura fofinho com cobertura' -> 'bolo de cenoura'"""
    t = sem_acento(sem_dono(nome).lower()).replace("-", " ")
    t = re.split(r"\s+(?:com|c/|recheado|recheada|coberto|coberta|-|\(|:)\s+|\(", t)[0]
    t = EXPRESSOES.sub(" ", t)
    palavras = [_singular(w) for w in re.sub(r"[^a-z ]", " ", t).split() if w not in ADJETIVOS]
    while palavras and palavras[-1] in ("de", "do", "da", "e", "com", "na", "no"): palavras.pop()
    return " ".join(palavras) or sem_acento(str(nome).lower())

def titulo_grupo(nome):
    """Mesmo corte do nome_base, mas mantendo os acentos do nome original."""
    t = re.split(r"\s+(?:com|c/|recheado|recheada|coberto|coberta|-|\(|:)\s+|\(", sem_dono(nome).strip(), flags=re.I)[0]
    palavras = [w for w in t.split() if sem_acento(w.lower()) not in ADJETIVOS]
    t = " ".join(palavras)
    t = re.sub(r"\s+(da|do|de)\s+(vovó|vó|mãe|mamãe|tia|sogra|vizinha|casa|fazenda|roça)\b.*$", "", t, flags=re.I)
    t = re.sub(r"\s+(II|III|IV|2|3)$", "", t)
    t = re.sub(r"\s+(de liquidificador|de panela.*|sem forno|(no|de|em|ao) micro.*|na airfryer|sem glúten|sem lactose|"
               r"sem açúcar|low carb|fit)\b", "", t, flags=re.I).strip(" -,")
    return t[:1].upper() + t[1:].lower() if t else str(nome)

df["grupo"] = df["nome"].apply(nome_base)

def agrupar(res, n_grupos, pular_grupos=0, max_variantes=MAX_VARIANTES):
    """Junta receitas do mesmo prato. Devolve os grupos (posições em res) já paginados.
    Quando um prato tem mais variantes que o limite, as que sobram viram outro card, mais para o fim."""
    grupos, extras = {}, []
    for i, g in enumerate(res["grupo"]):
        if g not in grupos: grupos[g] = [i]
        elif len(grupos[g]) < max_variantes: grupos[g].append(i)
        elif extras and res.iloc[extras[-1][0]]["grupo"] == g and len(extras[-1]) < max_variantes: extras[-1].append(i)
        else: extras.append([i])
    return (list(grupos.values()) + extras)[pular_grupos:pular_grupos + n_grupos]

PRATOS = {"Carnes", "Aves", "Peixes e frutos do mar", "Massas", "Prato Único"}
REFEICOES = [("Café da manhã", {"Bolos e tortas doces", "Lanches", "Bebidas"}),
             ("Almoço", PRATOS),
             ("Acompanhamento", {"Saladas, molhos e acompanhamentos", "Sopas"}),
             ("Jantar", PRATOS | {"Sopas"}),
             ("Sobremesa", {"Doces e sobremesas"})]

# No TudoGostoso, "Massas" também tem pães e massas de pizza: não servem como almoço ou jantar
NAO_E_PRATO = re.compile(r"^(pao|paes|paezinho|rosca|rosquinha|cuca|broa|biscoito|bolo|nozinho|trancinha|brioche|sonho|"
                         r"massa (de|para) (pizza|pastel|pao|torta|empada)|massa basica|massa caseira de pao|esfirra|esfiha|salgado)")
# Receita doce não é almoço nem jantar: leite condensado, chocolate, goiabada... ou muito açúcar
_DOCE = re.compile(r"leite condensado|chocolate|achocolatado|goiabada|doce de leite|marshmallow|granulado|chantilly")
# Em "Lanches" o TudoGostoso mistura salgados de festa e até arroz: no café entram só os lanches de manhã
CAFE = re.compile(r"\b(pao|paes|paozinho|sanduiche|misto|tapioca|panqueca|omelete|crepe|cuscuz|torrada|biscoito|bolacha|"
                  r"waffle|ovo|ovos|vitamina|suco|cafe|cappuccino|granola|mingau|cuca|broa|rosca|bisnaguinha|queijo quente|"
                  r"crepioca|pao de queijo|muffin|cookie|wrap|beiju|bolinho de chuva)\b")
def _eh_doce(itens):
    for it in itens:
        nomes = " ".join(n for v in it["vagas"] for n in v)
        if _DOCE.search(nomes): return True
        if "acucar" in nomes.split() and it["unidade"] in ("xicara", "copo", "kg") : return True
    return False
df["doce"] = df["itens"].apply(_eh_doce)

# Teto de cada refeição (fração da meta), para o café não comer a meta do jantar
TETO_REFEICAO = {"Café da manhã": .25, "Almoço": .40, "Acompanhamento": .15, "Jantar": .35, "Sobremesa": .20}

def montar_cardapio(termos, meta_kcal=2000, max_faltantes=MAX_FALTANTES, **filtros):
    escolhas, grupos_usados, total = [], set(), 0.0
    for nome, cats in REFEICOES:
        cands = buscar(termos, k=40, max_faltantes=max_faltantes, categorias=cats, **filtros)
        cands = cands[cands["kcal_porcao"].notna()]
        escolhida = None
        for _, r in cands.iterrows():
            if r["grupo"] in grupos_usados or total + r["kcal_porcao"] > meta_kcal: continue
            if r["kcal_porcao"] > TETO_REFEICAO[nome] * meta_kcal: continue
            if nome == "Café da manhã" and not (r["doce"] or CAFE.search(sem_acento(r["nome"].lower()))): continue
            if nome in ("Almoço", "Jantar") and (NAO_E_PRATO.search(sem_acento(r["nome"].lower())) or r["doce"]
                                                 or r["kcal_porcao"] < 150): continue
            escolhida = r; break
        if escolhida is not None:
            total += escolhida["kcal_porcao"]; grupos_usados.add(escolhida["grupo"])
        escolhas.append((nome, escolhida))
    return escolhas, total

print(Counter(df["grupo"]).most_common(8))
for n in ["Bolo de cenoura fofinho", "Bolo de Cenoura da Vovó com cobertura de chocolate", "Pão de queijo de liquidificador",
          "Mousse de maracujá fácil", "Strogonoff de frango rápido"]:
    print(f"  {n:52s} → {nome_base(n):22s} | {titulo_grupo(n)}")
escolhas, total = montar_cardapio(interpretar("ovo, leite, farinha de trigo, açúcar, frango, arroz, tomate, banana"), 2000,
                                  tenho_despensa=True)
for nome, r in escolhas:
    print(f"  {nome:16s} {r['nome'] if r is not None else '—'} ({r['kcal_porcao'] if r is not None else 0:.0f} kcal)")
print(f"  Total: {total:.0f} kcal")

## 8. O LLM: Qwen3.5-2B (o "G" do RAG)

- Modelo aberto da Alibaba, licença Apache-2.0, **2 bilhões de parâmetros**, escreve bem em português.
- Na T4 roda em **float32** (a T4 não acelera bfloat16); em GPUs mais novas, bfloat16.
- Modo **sem "pensamento"** (`enable_thinking=False`): responde mais rápido.
- **Sem ajuste fino:** o comportamento vem das regras do prompt. Ele recebe só as receitas que a busca encontrou e escreve a conversa: o que recomenda, por quê, o que comprar e uma dica. **Os cards (ingredientes, passos, nota) vêm direto da base**, então o modelo não tem como inventar uma receita.

In [ ]:
# ============ 8. LLM: Qwen3.5-2B ============
import transformers
from transformers import AutoProcessor

if USAR_LLM:
    usa_bf16 = GPU and torch.cuda.get_device_capability(0)[0] >= 8      # T4 = 7.5 -> float32
    DTYPE = torch.bfloat16 if usa_bf16 else torch.float32
    ClasseModelo = (getattr(transformers, "AutoModelForMultimodalLM", None)
                    or getattr(transformers, "AutoModelForImageTextToText"))
    inicio = time.time()
    proc = AutoProcessor.from_pretrained(MODELO_LLM)
    modelo = ClasseModelo.from_pretrained(MODELO_LLM, dtype=DTYPE, device_map="auto")
    modelo.eval()
    print(f"{MODELO_LLM} carregado em {(time.time() - inicio) / 60:.1f} min · {DTYPE} · "
          f"{sum(p.numel() for p in modelo.parameters()) / 1e9:.2f} bi de parâmetros")

def gerar(msgs, max_new_tokens=TOKENS_LLM):
    inputs = proc.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=True,
                                      return_tensors="pt", enable_thinking=False).to(modelo.device)
    with torch.no_grad():
        out = modelo.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=0.7, top_p=0.8, top_k=20, repetition_penalty=1.05)
    texto = proc.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", texto, flags=re.S).strip()

def usar_llm():
    return USAR_LLM and "modelo" in globals()

SISTEMA = (
    "Você é o Remy, um ratinho chef simpático que conversa em português do Brasil.\n"
    "O sistema já buscou receitas reais do site TudoGostoso e vai mostrá-las em cartões logo abaixo da sua mensagem "
    "(com tempo, nota, calorias, ingredientes e modo de preparo).\n"
    "Escreva uma mensagem curta (3 a 6 frases) que:\n"
    "1. responde ao que o usuário pediu;\n"
    "2. destaca a melhor opção e por quê (usa mais do que ele tem, é rápida ou é muito bem avaliada);\n"
    "3. diz o que vale comprar, se faltar algo;\n"
    "4. termina com uma dica prática de cozinha.\n"
    "Regras: fale SÓ das receitas fornecidas, com o nome exatamente como está. Não invente receitas, ingredientes ou "
    "números. Não repita o modo de preparo (ele está nos cartões). Use **negrito** no nome das receitas. Sem títulos. "
    "Se uma receita tiver variantes, diga que dá para escolher a variante no próprio cartão. "
    "As calorias são estimativas: se falar delas, diga 'cerca de'."
)

if usar_llm():
    msgs = [{"role": "system", "content": [{"type": "text", "text": SISTEMA}]},
            {"role": "user", "content": [{"type": "text", "text": "Pedido: o que faço com frango e arroz?\nReceitas encontradas:\n"
             "- Receita: Arroz com frango · 40 min · nota 4.7 (1200 avaliações) · cerca de 420 kcal por porção · "
             "tem: frango, arroz · falta: 1 cenoura\n\nEscreva sua mensagem."}]}]
    inicio = time.time(); print(gerar(msgs)); print(f"\n({time.time() - inicio:.0f} s)")

## 9. A conversa: ponte com o site

O site manda um JSON (`mensagem`, `ingredientes`, `modo`, `meta_calorias`, `historico`) e recebe outro (`texto`, `receitas`, `cardapio`). Aqui o Remy:

1. **Acha os ingredientes:** dos chips do site; se vierem vazios, do texto (*"o que faço com frango e arroz?"*); se ainda não tiver, da mensagem anterior da conversa.
2. **Quanto pode faltar:** a opção *Pode faltar* do site (nada, até 2, até 5 ou tanto faz) ou o texto (*"só com o que tenho"*, *"posso comprar"*).
3. **Tira filtros do texto:** *"rápido"* ou *"30 min"*, *"vegetariano"*, *"vegano"*, *"sem glúten"*, *"sem lactose"*, *"sem camarão"*, *"sobremesa"*, *"bolo"*, *"lanche"*, *"até 400 kcal"*, *"tenho o básico"*.
4. **Pergunta** o tipo dos ingredientes ambíguos e avisa as correções de digitação.
5. **"Mais opções"** mostra as próximas receitas, sem repetir, mantendo os filtros.
6. Monta os **cards completos**: ingredientes por parte, modo de preparo, tempo, porções, nota real, calorias estimadas e o crédito ao TudoGostoso.
7. O **Qwen** escreve a mensagem do Remy.

In [ ]:
# ============ 9. CONVERSA E PONTE COM O SITE ============
# ---------- filtros que saem do texto ----------
PALAVRAS_DIETA = {"vegetariano": r"vegetarian", "vegano": r"vegan", "sem glúten": r"sem gluten|celiac",
                  "sem lactose": r"sem lactose|intoleran"}
PALAVRAS_CATEGORIA = [(r"cafe da manha|desjejum", {"Bolos e tortas doces", "Lanches", "Bebidas"}),
                      (r"sobremesa|\bdoces?\b|docinho", {"Doces e sobremesas", "Bolos e tortas doces"}),
                      (r"\bbolos?\b|\btortas? doces?\b", {"Bolos e tortas doces"}),
                      (r"salada|acompanhamento|molho", {"Saladas, molhos e acompanhamentos"}),
                      (r"\bmassas?\b|macarronada", {"Massas"}), (r"lanche|petisco|salgadinho", {"Lanches"}),
                      (r"frutos do mar|peixes?\b", {"Peixes e frutos do mar"}), (r"bebida|\bsucos?\b|drinks?", {"Bebidas"}),
                      (r"\bsopas?\b|caldinho", {"Sopas"}), (r"prato principal|almoco|jantar|janta", {
                          "Carnes", "Aves", "Peixes e frutos do mar", "Massas", "Prato Único"})]
PALAVRAS_PEDIDO = (r"\b(o que|oque|que|qual|quais|da pra|da para|posso|fazer|faco|receitas?|tenho|com|me|sugere|sugira|"
                   r"sugestao|algo|alguma|coisa|outra|outro|opcao|opcoes|mais|quero|queria|gostaria|pode|ser|para|pra|"
                   r"uma|um|so|aqui|em casa|na geladeira|hoje|agora|por favor|oi|ola|bom dia|boa tarde|boa noite|"
                   r"obrigad[oa]|rapid[oa]s?|rapidinh[oa]|minutos?|min|kcal|calorias?|ate|\d+|vegetarian[oa]s?|"
                   r"vegan[oa]s?|gluten|lactose|sobremesas?|doces?|docinhos?|bolos?|jantar|janta|almoco|cafe da manha|"
                   r"lanches?|sopas?|saladas?|petiscos?|bebidas?|acompanhamentos?|massas?|leve|saudavel|basico|despensa)\b")

def filtros_do_texto(msg):
    t = sem_acento((msg or "").lower())
    f = {"tempo_max": None, "dietas": [], "categorias": None, "evitar": [], "despensa": False, "kcal_max": None,
         "max_faltantes": None}
    if re.search(r"so com o que (eu )?tenho|nada faltando|sem precisar comprar|sem comprar", t): f["max_faltantes"] = 0
    elif re.search(r"posso comprar|pode faltar|vou (ao|no) mercado|tanto faz|nao precisa ser so", t): f["max_faltantes"] = 99
    m = re.search(r"(\d+)\s*(?:min|minutos)", t)
    if m: f["tempo_max"] = int(m.group(1))
    elif re.search(r"rapid|pressa|corrid", t): f["tempo_max"] = 30
    m = re.search(r"(\d+)\s*(?:kcal|calorias)", t)
    if m: f["kcal_max"] = int(m.group(1))
    elif re.search(r"\bleve\b|light|poucas calorias", t): f["kcal_max"] = 350
    for nome, regex in PALAVRAS_DIETA.items():
        if re.search(regex, t): f["dietas"].append(nome)
    for regex, cats in PALAVRAS_CATEGORIA:
        if re.search(regex, t): f["categorias"] = cats; break
    for m in re.finditer(r"(?:sem|nao gosto de|nao como|alergi[ac] a|alergico a|evitar|tirando)\s+([a-z ]+?)(?:[,.;!?]|$| e )", t):
        termo = m.group(1).strip()
        if termo and termo not in ("gluten", "lactose", "acucar") and not termo.startswith(("pouco", "sal ")):
            f["evitar"].append(termo)
    if re.search(r"tenho (o )?basico|despensa", t): f["despensa"] = True
    return f

def juntar_filtros(lista):
    f = filtros_do_texto("")
    for g in lista:
        if g["tempo_max"]: f["tempo_max"] = min(f["tempo_max"] or 10**6, g["tempo_max"])
        if g["kcal_max"]: f["kcal_max"] = min(f["kcal_max"] or 10**6, g["kcal_max"])
        f["dietas"] = list(dict.fromkeys(f["dietas"] + g["dietas"]))
        f["categorias"] = g["categorias"] or f["categorias"]
        f["evitar"] = list(dict.fromkeys(f["evitar"] + g["evitar"]))
        f["despensa"] |= g["despensa"]
        if g.get("max_faltantes") is not None: f["max_faltantes"] = g["max_faltantes"]
    return f

def _flex(regex):
    """Deixa a regex sem acento aceitar o texto com acento: 'acucar' também casa com 'açúcar'."""
    for a, b in (("a", "[aáàâã]"), ("e", "[eéê]"), ("i", "[ií]"), ("o", "[oóôõ]"), ("u", "[uú]"), ("c", "[cç]")):
        regex = re.sub(rf"(?<![\\\[]){a}(?![^\[]*\])", b, regex)
    return regex
PEDIDO_FLEX = re.compile(_flex(PALAVRAS_PEDIDO))

def ingredientes_do_texto(msg):
    """'o que faço com frango e arroz?' -> ['frango', 'arroz'] (mantém os acentos do usuário)"""
    t = (msg or "").lower()
    t = re.split(_flex(r"\b(?:sem|nao gosto|alergi|evitar|tirando)\b"), t)[0]
    t = PEDIDO_FLEX.sub(" ", t)
    t = re.sub(r"[?!.]", " ", t)
    return [x.strip() for x in separar_itens(t) if len(x.strip()) > 1]

def ingredientes_validos_do_texto(msg):
    itens = ingredientes_do_texto(msg)
    return [t for t in interpretar(", ".join(itens)) if t["ids"]] if itens else []

MAIS = re.compile(r"\b(mais|outr[ao]s?|diferentes?|de novo|nenhuma dessas|nao gostei|proxim[ao]s?)\b")
def eh_pedido_de_mais(txt):
    return bool(MAIS.search(sem_acento((txt or "").lower()))) and not ingredientes_validos_do_texto(txt)

def cadeia_do_historico(historico):
    """Mensagens do usuário desde a última que trouxe ingredientes: (ingredientes, [textos depois dela])."""
    usuario = [h["content"] for h in (historico or []) if h.get("role") == "user" and h.get("content")]
    for i in range(len(usuario) - 1, -1, -1):
        termos = ingredientes_validos_do_texto(usuario[i])
        if termos: return termos, usuario[i:]
    return [], []

def perguntas(termos):
    out = []
    for t in termos:
        if t["aviso"]: out.append(t["aviso"])
        if t["ambiguo"]:
            padrao, opcoes, _ = AMBIGUOS[t["ambiguo"]]
            out.append(f"🧂 Que tipo de **{t['digitado']}** você tem? Por enquanto considerei {padrao}. "
                       f"Se for outro, me diga, por exemplo: *{', '.join(opcoes[1:5])}*.")
    return out

# Grupos de alergia: "frutos do mar" vira a lista de ingredientes que entram nele
GRUPOS_ALERGIA = {
    "frutos do mar": ["camarão", "lula", "polvo", "marisco", "mexilhão", "siri", "caranguejo", "lagosta", "kani"],
    "peixe": ["peixe", "atum", "sardinha", "bacalhau", "salmão", "tilápia", "merluza", "pescada", "anchova"],
    "castanhas": ["castanha de caju", "castanha do pará", "noz", "amêndoa", "avelã", "pistache", "macadâmia"],
    "castanha": ["castanha de caju", "castanha do pará", "noz", "amêndoa", "avelã", "pistache", "macadâmia"],
    "amendoim": ["amendoim", "paçoca", "pasta de amendoim"],
    "ovo": ["ovo", "gema", "clara"], "ovos": ["ovo", "gema", "clara"],
    "carne de porco": ["bacon", "lombo", "pernil", "linguiça", "calabresa", "presunto", "toucinho", "costelinha de porco", "paio", "salame"],
    "soja": ["soja", "shoyu", "tofu", "proteína de soja"],
}

TIPO_RECEITA = {"cafe": {"Bolos e tortas doces", "Lanches", "Bebidas"},
                "principal": {"Carnes", "Aves", "Peixes e frutos do mar", "Massas", "Prato Único", "Sopas"},
                "sobremesa": {"Doces e sobremesas", "Bolos e tortas doces"}}

# ---------- quantidade que o usuário tem x o que a receita pede ----------
UNIDADE_SITE = {"un": None, "g": "g", "kg": "kg", "ml": "ml", "l": "l", "xicara": "xicara", "colher sopa": "colher sopa",
                "lata": "lata", "caixa": "caixa", "pacote": "pacote"}

def _fmt(x):
    return f"{x:.0f}" if abs(x - round(x)) < 1e-6 else f"{x:.1f}".replace(".", ",")

def checar_quantidades(rid, termos, quantidades):
    """Compara a quantidade da geladeira com a da receita. Devolve avisos como
    'a receita pede 6 ovos; você tem 3'."""
    if not quantidades: return []
    avisos = []
    porTermo = []
    for t in termos:
        q = quantidades.get(t["digitado"]) or quantidades.get(t["digitado"].strip())
        if q and q.get("qtd"): porTermo.append((t, float(q["qtd"]), UNIDADE_SITE.get(q.get("unidade"), None)))
    if not porTermo: return []
    for item in df.at[rid, "itens"]:
        if not item["vagas"] or item["qtd"] is None: continue
        ids_item = {idx[n] for v in item["vagas"] for n in v if n in idx}
        for t, qtd, un in porTermo:
            if not (t["ids"] & ids_item): continue
            nome = item["vagas"][0][0]
            mesma = (un or None) == (item["unidade"] if item["unidade"] != "unidade" else None)
            if mesma:
                tenho, pede = qtd, item["qtd"]
            else:
                tenho = gramas({"qtd": qtd, "unidade": un}, nome)
                pede = gramas(item, nome)
            if tenho is not None and pede and tenho < pede * 0.85:
                avisos.append(f"{item['texto']} (você tem {_fmt(qtd)}{' ' + (un or 'un') if un else ''})")
            break
    return avisos

# ---------- cards ----------
def _nota(r):
    return round(float(r["nota"]), 1) if pd.notna(r["nota"]) and r["n_avaliacoes"] > 0 else None

def card(r):
    secoes = {}
    for item in r["itens"]:
        secoes.setdefault(item.get("secao") or "", []).append(item["texto"])
    return {
        "nome": r["nome"], "categoria": r["categoria"],
        "minutos": int(r["minutos"]) if pd.notna(r["minutos"]) else None,
        "porcoes": int(r["porcoes"]) if pd.notna(r["porcoes"]) else None,
        "nota": _nota(r), "avaliacoes": int(r["n_avaliacoes"]),
        "calorias": int(r["kcal_porcao"]) if pd.notna(r["kcal_porcao"]) else None, "calorias_aprox": True,
        "tem": list(r["tem"]), "falta": list(r["falta"]), "pouco": list(r.get("pouco", []) or []),
        "ingredientes": [{"secao": s, "itens": l} for s, l in secoes.items()],
        "passos": [p["texto"] for p in r["passos"]],
        "fonte": "TudoGostoso",
    }

def cards_agrupados(res, grupos):
    out = []
    for g in grupos:
        vs = [card(res.iloc[j]) for j in g]
        c = dict(vs[0])
        if len(vs) > 1:
            c["grupo"] = titulo_grupo(res.iloc[g[0]]["nome"])
            nomes = [v["nome"] for v in vs]
            for v in vs:
                v["rotulo"] = v["nome"] if nomes.count(v["nome"]) == 1 else f"{v['nome']} · {v['minutos']} min"
            c["variantes"] = vs
        out.append(c)
    return out

# ---------- texto do Remy ----------
def _historico_llm(historico, n=4):
    return [{"role": h["role"], "content": [{"type": "text", "text": h["content"][:600]}]}
            for h in (historico or [])[-n:] if h.get("role") in ("user", "assistant") and h.get("content")]

def _resumo(c):
    partes = [f"- {c.get('refeicao', 'Receita')}: {c['nome']}"]
    if c["minutos"]: partes.append(f"{c['minutos']} min")
    partes.append(f"nota {c['nota']} ({c['avaliacoes']} avaliações)" if c["nota"] else "sem avaliações")
    if c["calorias"]: partes.append(f"cerca de {c['calorias']} kcal por porção")
    if c.get("variantes"): partes.append(f"{len(c['variantes'])} variantes de {c['grupo']}")
    if c.get("pouco"): partes.append(f"pode faltar quantidade: {'; '.join(c['pouco'])}")
    partes += [f"tem: {', '.join(c['tem']) or 'nenhum'}", f"falta: {'; '.join(c['falta']) or 'nada'}"]
    return " · ".join(partes)

def texto_remy(pedido, cards, historico, extra=""):
    if not cards: return ""
    if usar_llm():
        usuario = (f"Pedido do usuário: {pedido}\n{extra}\nReceitas encontradas:\n"
                   + "\n".join(_resumo(c) for c in cards[:3]) + "\n\nEscreva sua mensagem.")
        msgs = ([{"role": "system", "content": [{"type": "text", "text": SISTEMA}]}] + _historico_llm(historico)
                + [{"role": "user", "content": [{"type": "text", "text": usuario}]}])
        try:
            return gerar(msgs)
        except Exception as e:
            print("⚠️ LLM falhou:", e)
    m = cards[0]
    return (f"Separei {len(cards)} opções. A melhor é **{m['nome']}**"
            + (f", e só falta: {'; '.join(m['falta'])}." if m["falta"] else ", e você já tem tudo!"))

# ---------- dica para a receita escolhida ----------
def dica_receita(rec):
    """Uma dica curta para a receita que o usuário escolheu. Com o LLM, o Qwen escreve; sem ele, regras simples."""
    ingredientes = [i for g in rec.get("ingredientes", []) for i in g.get("itens", [])]
    passos = rec.get("passos", [])
    if usar_llm():
        texto = (f"Receita escolhida: {rec.get('nome')}\nIngredientes:\n- " + "\n- ".join(ingredientes[:25])
                 + "\nModo de preparo:\n" + "\n".join(f"{i + 1}. {p}" for i, p in enumerate(passos[:20]))
                 + "\n\nDê UMA dica prática para essa receita dar certo, em 1 ou 2 frases. Baseie-se só nesses "
                   "ingredientes e passos. Não invente ingredientes nem mude a receita. Comece com 'Dica do Remy:'.")
        msgs = [{"role": "system", "content": [{"type": "text", "text": "Você é o Remy, um ratinho chef que fala português do Brasil."}]},
                {"role": "user", "content": [{"type": "text", "text": texto}]}]
        try:
            return gerar(msgs, max_new_tokens=120)
        except Exception as e:
            print("⚠️ LLM falhou na dica:", e)
    tudo = sem_acento(" ".join(passos + ingredientes).lower())
    regras = [(r"forno", "preaqueça o forno uns 10 minutos antes e não abra a porta na primeira metade do tempo."),
              (r"claras? em neve", "bata as claras numa tigela bem seca e sem gema nenhuma, senão elas não sobem."),
              (r"fermento", "o fermento vai por último e é só misturado devagar, sem bater."),
              (r"panela de pressao", "espere a pressão sair sozinha antes de abrir a panela."),
              (r"fritar|frite|oleo quente", "teste o óleo com um pedacinho antes: se borbulhar em volta, está na temperatura."),
              (r"gelad|geladeira", "respeite o tempo de geladeira: é ele que dá a consistência.")]
    for regex, dica in regras:
        if re.search(regex, tudo): return "Dica do Remy: " + dica
    return "Dica do Remy: separe e meça todos os ingredientes antes de começar. A receita fica bem mais tranquila."

# ---------- a função que o site chama ----------
NOMES_FILTRO = {"ingredientes faltando": "ingredientes faltando (aumente em \"Pode faltar\")", "tempo": "tempo", "categoria": "categoria",
                "calorias": "calorias", "ingredientes a evitar": "ingredientes a evitar", "nota": "nota mínima"}

def responder_site(pedido_json):
    try:
        pedido = json.loads(pedido_json) if isinstance(pedido_json, str) else pedido_json
    except Exception:
        pedido = pedido_json
    if not isinstance(pedido, dict):
        pedido = {"mensagem": str(pedido)}
    if pedido.get("acao") == "ingredientes":           # o site pede a lista para o "Toque para adicionar"
        return json.dumps({"ingredientes": INGREDIENTES_SITE}, ensure_ascii=False)
    if pedido.get("acao") == "dica":                   # o usuário escolheu uma receita
        return json.dumps({"dica": dica_receita(pedido.get("receita") or {})}, ensure_ascii=False)
    msg = (pedido.get("mensagem") or "").strip()
    quantidades = pedido.get("quantidades") or {}
    historico = pedido.get("historico") or []
    modo = pedido.get("modo") or "receitas"
    f, pular, avisos = filtros_do_texto(msg), 0, []
    if pedido.get("tipo") in TIPO_RECEITA:              # tipo escolhido no site: café da manhã, prato principal ou sobremesa
        f["categorias"] = TIPO_RECEITA[pedido["tipo"]]

    # 1. ingredientes: chips > texto > conversa anterior
    chips = [str(c).strip() for c in (pedido.get("ingredientes") or []) if str(c).strip()]
    if chips:
        termos = interpretar(", ".join(chips))
        avisos = perguntas(termos)
    else:
        termos = ingredientes_validos_do_texto(msg)
        avisos = perguntas(termos)
        if not termos:                                    # continuação: "mais opções", "e algo vegetariano?"
            termos, cadeia = cadeia_do_historico(historico)
            f = juntar_filtros([filtros_do_texto(c) for c in cadeia] + [f])
            if eh_pedido_de_mais(msg):
                seguidas = 0
                for c in reversed(cadeia[1:]):
                    if not eh_pedido_de_mais(c): break
                    seguidas += 1
                pular = N_RECEITAS * (seguidas + 1)
    validos = [t for t in termos if t["ids"]]
    if not validos:
        return json.dumps({"texto": "\n\n".join(avisos) or "Oi! Sou o Remy 🐭 Me conta o que tem na sua geladeira, "
                           "separado por vírgula. Por exemplo: *frango, arroz, tomate*."}, ensure_ascii=False)

    entendi = "Entendi: " + ", ".join(t["nome"] for t in validos) + "."
    perfil = pedido.get("perfil") or {}
    if perfil.get("dietas") or perfil.get("evitar"):        # restrições salvas no perfil valem sempre
        evitar_perfil = []
        for e in perfil.get("evitar") or []:
            chave = sem_acento(str(e).lower().strip())
            if chave in ("leite", "lactose"): f["dietas"].append("sem lactose")
            elif chave in ("gluten", "trigo"): f["dietas"].append("sem glúten")
            else: evitar_perfil += GRUPOS_ALERGIA.get(chave, [e])
        f["dietas"] = list(dict.fromkeys(f["dietas"] + [d for d in perfil.get("dietas") or [] if d in TEM_DIETA]))
        f["evitar"] = list(dict.fromkeys(f["evitar"] + evitar_perfil))
        entendi += f" Considerei seu perfil: {', '.join([*(perfil.get('dietas') or []), *('sem ' + str(e) for e in perfil.get('evitar') or [])])}."
    excluir = interpretar(", ".join(f["evitar"])) if f["evitar"] else []
    # Quantos ingredientes podem faltar: o texto ("posso comprar") > a opção "Pode faltar" do site > padrão
    opcao = pedido.get("max_faltantes")
    max_falt = f["max_faltantes"] if f["max_faltantes"] is not None else (
        99 if opcao == "livre" else int(opcao) if str(opcao).isdigit() else MAX_FALTANTES)
    filtros = dict(tempo_max=f["tempo_max"], dietas=f["dietas"], kcal_max=f["kcal_max"], excluir=excluir,
                   tenho_despensa=f["despensa"], max_faltantes=max_falt)
    pedido_txt = msg or ("Montar um cardápio do dia" if modo == "cardapio" else "O que posso cozinhar?")

    # 2. cardápio do dia
    if modo == "cardapio":
        meta = int(pedido.get("meta_calorias") or 2000)
        escolhas, total = montar_cardapio(validos, meta, **filtros)
        cards = []
        for nome, r in escolhas:
            if r is not None:
                r = r.copy(); r["pouco"] = checar_quantidades(r["rid"], validos, quantidades)
                c = card(r); c["refeicao"] = nome; cards.append(c)
        vazias = [n for n, r in escolhas if r is None]
        if vazias:
            avisos.append("😕 Não achei opção para: " + ", ".join(vazias) + ". Que tal me dizer mais ingredientes?")
        texto = texto_remy(pedido_txt, cards, historico, f"Meta do dia: {meta} kcal. Total: cerca de {total:.0f} kcal.")
        return json.dumps({"texto": "\n\n".join([entendi] + avisos + [texto]).strip(), "receitas": cards,
                           "cardapio": {"meta_calorias": meta, "total_calorias": round(total), "calorias_aprox": True,
                                        "itens": [{"refeicao": c["refeicao"], "nome": c["nome"], "calorias": c["calorias"]}
                                                  for c in cards]}}, ensure_ascii=False)

    # 3. receitas (busca mais candidatas e agrupa as variantes)
    res, diag = buscar(validos, k=12 * (N_RECEITAS + pular), categorias=f["categorias"], retornar_diag=True, **filtros)
    if len(res) and quantidades:
        res["pouco"] = [checar_quantidades(rid, validos, quantidades) for rid in res["rid"]]
    grupos = agrupar(res, N_RECEITAS, pular_grupos=pular)
    if not grupos and pular:
        return json.dumps({"texto": entendi + "\n\nJá te mostrei todas as receitas que combinam com esses ingredientes "
                           "e filtros. 🐭 Me diga mais um ingrediente, ou tire algum filtro, que eu acho outras."},
                          ensure_ascii=False)
    if not grupos:
        dicas = [f"sem o filtro de **{NOMES_FILTRO.get(k, k)}** eu acharia {q:,} receitas"
                 for k, q in sorted(diag.items(), key=lambda x: -x[1]) if q > 0 and k != "usar o que você tem"][:2]
        return json.dumps({"texto": "\n\n".join([entendi] + avisos + ["😕 Não achei nenhuma receita com tudo isso."
                           + (" " + "; ".join(dicas) + "." if dicas else " Tente me dizer mais ingredientes.")])},
                          ensure_ascii=False)
    cards = cards_agrupados(res, grupos)
    texto = texto_remy(pedido_txt, cards, historico)
    return json.dumps({"texto": "\n\n".join([entendi] + avisos + [texto]).strip(), "receitas": cards}, ensure_ascii=False)

# teste com o mesmo pedido que o site manda
teste = {"mensagem": "", "ingredientes": ["frango", "arroz", "tomate", "cebola", "alho"], "modo": "receitas",
         "meta_calorias": 2000, "historico": []}
inicio = time.time()
r = json.loads(responder_site(json.dumps(teste)))
print(f"Resposta em {time.time() - inicio:.0f} s\n\n{r['texto']}\n")
for c in r["receitas"]:
    print(f"- {c.get('grupo', c['nome'])} · {c['minutos']} min · nota {c['nota']} · ≈{c['calorias']} kcal · falta: {c['falta']}")

## 10. Ligando o site

Rode a célula abaixo. Ela abre um servidor Gradio com o endpoint **`/responder`** e mostra um link público parecido com `https://xxxxxxxx.gradio.live`.

No site do Remy.IA:
1. Clique em **Conectar modelo**.
2. Em *Fonte das respostas*, escolha **Gradio: Colab ou Hugging Face Space**.
3. Cole o link `https://....gradio.live` e use o endpoint `/responder`.
4. Salve e mande os ingredientes.

O link vale enquanto este Colab estiver aberto. Se a sessão cair, rode de novo e cole o link novo no site.

In [ ]:
# ============ 10. SERVIDOR PARA O SITE ============
import gradio as gr

demo = gr.Interface(
    fn=responder_site,
    inputs=gr.Textbox(label="pedido_json", lines=4, value=json.dumps(teste, ensure_ascii=False)),
    outputs=gr.Textbox(label="resposta_json", lines=12),
    title="Remy.IA · API do site",
    description="O site do Remy.IA chama este endpoint (/responder). Dá para testar aqui também.",
    api_name="responder",
)
demo.queue(default_concurrency_limit=1)      # uma resposta por vez na GPU
demo.launch(share=True, debug=False)
print("\nCopie o link https://....gradio.live acima e cole no site em Conectar modelo.")